# Solar Filament Segmentation 2026 — assembled from `src/`

Generated by `scripts/build_notebook.py` — **do not edit here**; the repo is the source
of truth. One cell per module: config → data → rle → metric → postprocess → dataset →
model → evaluate → detector → refiner → cascade → pipeline, then the runner.

**Two modes, decided by what is attached.** With a dataset of `fold*_best.pt`
checkpoints attached: inference only. Without one: train grouped folds until the time
budget, save checkpoints + OOF prob maps, sweep post-processing on OOF, write the
fold-ensemble `submission.csv`. A placeholder submission is written first.

Metric: Panoptic Quality (IoU > 0.5 match), reproduced in `metric` from the organiser's
self-evaluation notebook (azimahmadzadeh/self-evaluation-notebook).


In [ ]:
# --- bootstrap: dependencies ------------------------------------------------------------
import os, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # before any CUDA allocation
for mod, pkg in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("ultralytics", "ultralytics"),
                 ("rfdetr", "rfdetr[train]"), ("pytorch_lightning", "rfdetr[train]")):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import segmentation_models_pytorch as smp
import ultralytics
import rfdetr
print("smp", smp.__version__, "ultralytics", ultralytics.__version__, "rfdetr", getattr(rfdetr, "__version__", "?"))


In [ ]:
# === src/config.py ===
"""Shared constants, environment discovery and logging.

The same modules run in two places: this repo (Apple-silicon Mac, smoke tests
and post-processing sweeps) and a Kaggle GPU kernel assembled from them by
scripts/build_notebook.py. Everything path-like is resolved here.
"""
from __future__ import annotations

import os
import random
import time
from pathlib import Path

import numpy as np
import torch

T0 = time.time()
SEED = 2026


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


ON_KAGGLE = Path("/kaggle/working").is_dir()
try:
    REPO = Path(__file__).resolve().parents[1]
except NameError:  # notebook cell: no __file__
    REPO = Path.cwd()
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO
CACHE = Path("/tmp/filament_cache") if ON_KAGGLE else REPO / "cache"
RUNS = WORK if ON_KAGGLE else REPO / "runs"

COMP = "filament-segmentation-2026"
DATA_DIRNAME = "MAGFiLO_1.0_Kaggle_2026"
ANN_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"


def find_data_root() -> Path:
    env = os.environ.get("FILAMENT_DATA")
    cands = [Path(env)] if env else []
    cands += [REPO / "data" / DATA_DIRNAME,
              Path("/kaggle/input/competitions") / COMP / DATA_DIRNAME,
              Path("/kaggle/input") / COMP / DATA_DIRNAME]
    for c in cands:
        if (c / "train" / "train_images").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for pat in (f"*/{DATA_DIRNAME}/train/{ANN_NAME}", f"*/*/{DATA_DIRNAME}/train/{ANN_NAME}"):
            hits = sorted(base.glob(pat))
            if hits:
                return hits[0].parent.parent
    raise FileNotFoundError("competition data not found (set FILAMENT_DATA)")


# --- image / crops --------------------------------------------------------------------
H = W = 2048               # native GONG frame; the metric is computed at this size
IMG_SIZE = 1024            # working resolution. v1-v3: 1024 (thickness ~13 px native -> ~6 px);
                           # v4 tried native 2048: FAIL (-0.019 OOF, SQ unchanged) - see PORTING
CROP = 512                 # training crop taken from the IMG_SIZE image
POSITIVE_BIAS = 0.7        # fraction of crops centred on a filament pixel

# --- model / training -----------------------------------------------------------------
ARCH = "unet"              # smp architecture: unet | unetpp | fpn | deeplab | manet
ENCODER = "resnet34"
EPOCHS = 32                # v1 (20 epochs): still rising; v2/v3 (32): converged by 25-31
BATCH = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 2.0           # BCE positive weight; ~0.3 % of pixels are filament
SAMPLES_PER_FILE = 4       # crops drawn per file per epoch
TARGET_MODE = "random"     # "random": one annotator record per crop at random (v1/v2);
                           # "medoid": the record with the highest mean mask-IoU to the
                           # other records of the same file (consensus target, rung 5)
WORKERS = 3 if ON_KAGGLE else 4

# --- folds / budget -------------------------------------------------------------------
N_FOLDS = 5                # grouped by physical jpeg (annotator records never split)
MAX_FOLDS = "auto"         # "auto" trains folds until TIME_BUDGET would be exceeded
START_FOLD = 0
TIME_BUDGET = 8.5 * 3600   # seconds; Kaggle GPU sessions are capped at 12 h

# --- post-processing / selection ------------------------------------------------------
EPOCH_THR = 0.5            # threshold used for per-epoch model selection (no TTA)
MIN_AREA = 150             # component area (native px) below which a prediction is dropped
THRESHOLDS = (0.55, 0.65, 0.75)         # final OOF sweep grid (v1 sweeps: plateau 0.6-0.7) ...
MIN_AREAS = (300, 450)                  # ... x this (v1: 300-400 best, 600 too aggressive) ...
MIN_SCORES = (0.8, 0.9)                 # ... x per-component mean-prob floor (v1 OOF: peak 0.90,
                                        #     +0.015 over none; 0.92 already collapses)
CLOSE_KERNEL = 5           # morphological closing before components (v1 OOF: +0.002)
TTA = True                 # 4-way flip TTA for OOF and test

# --- smoke mode -----------------------------------------------------------------------
DEBUG = os.environ.get("FILAMENT_DEBUG", "0") == "1"
MAX_STEPS = 0              # 0 = full epoch
EVAL_FILES = 0             # 0 = whole validation fold
if DEBUG:
    EPOCHS, MAX_STEPS, EVAL_FILES, MAX_FOLDS = 1, 20, 8, 2
    THRESHOLDS, MIN_AREAS, MIN_SCORES = (0.45, 0.55), (150,), (0.0,)
    WORKERS = 0            # macOS spawn cannot pickle notebook-namespace classes; Kaggle forks
    TTA = False            # 4 passes x 180 test images at 2048 is too slow for a smoke test

# --- rung 4: detector -> crop-refiner cascade ------------------------------------------
# PIPELINE selects the recipe: "semantic" = v1-v4 (U-Net prob map -> components);
# "cascade" = YOLO box detector -> per-box crop refiner -> one mask per detection.
PIPELINE = "cascade"
DET_ARCH = "rfdetr"         # v12: "rfdetr" (query-based, no NMS, DINOv2 backbone; RF100-VL small-object AP
                            # 35.8 vs YOLO11's 7.3) | "yolo" (v5-v11). Same union labels, folds, refiner.
RFDETR_SIZE = "medium"      # nano | small | medium | base (Apache-2.0 variants)
RFDETR_RES = 896            # must be divisible by patch 16 x 2 windows = 32 (and 64 for square resize);
                            # median filament 96 native px -> 42 px. v12a at 1024 + expanded scales (train sizes
                            # up to 1184) went OOM on the T4 at batch 2
RFDETR_EPOCHS = 25            # fine-tuned from the COCO-pretrained RF-DETR; 40 at 1024 px risks the 12 h cap
RFDETR_BATCH = 1             # per step (T4 16 GB); effective batch via accumulation
RFDETR_GRAD_ACCUM = 16      # effective batch 16 (RF-DETR recommended total)
RFDETR_EXPANDED_SCALES = False  # multi-scale jitter 800-1024 at res 896 (expanded would reach 1056)
RFDETR_LR = 1e-4
RFDETR_EVAL_INTERVAL = 5    # validation every N epochs (saves ~20 % of T4 time)
RFDETR_ACCELERATOR = "auto" # Lightning accelerator; the local Mac smoke test uses "cpu" (MPS AMP needs float64)
DET_MODEL = "yolo11s.pt"    # v10 tried yolo11m: WASH (0.4406 vs 0.4449, contested recall flat). COCO-pretrained
DET_IMGSZ = 1280            # v5-v8: 1024 on the cached PNGs (no resize). v9: 1280 from the native
                            # 2048 JPEGs (small filaments: consensus recall 0.79 < 64 px vs 0.89)
DET_SOURCE = "native"       # "cache": img{IMG_SIZE}.png (v5-v8) | "native": the 2048 JPEGs
DET_EPOCHS = 40             # YOLO epochs (v11 used 15 with views)
DET_BATCH = 8
DET_LABEL_RECORD = "union"  # YOLO labels are static files. "medoid": the consensus record (v5).
                            # "union": every record's filaments, deduplicated across annotators at
                            # mask IoU > 0.5, kept when drawn by >= DET_LABEL_MIN_SUPPORT of the
                            # file's annotators (v6; the pooled metric rewards predicting a filament
                            # iff support > PQ / (2 SQ) ~ 0.32, see PORTING 2026-09-19)
DET_LABEL_MIN_SUPPORT = 0.5
DET_EXTRA_VIEWS = 0         # v11 lever (FAIL, -0.011): archive views per TRAIN-fold file added to the detector's
                            # training list (val = the labelled frames only). 0 = off (v5-v10);
                            # N > 0 = up to N views per file from data/sequences_long/<stem>/ or the
                            # attached Kaggle dataset ishansharma13/gong-halpha-train-views-long:
                            # other competition sites at the same minute (ECC-aligned, labels
                            # unchanged) and the same site within DET_VIEWS_MAX_DT_H hours (labels
                            # rotated). Views are LUT-matched to the competition processing
                            # (src/views.py). Host ruling: sequences at training time only.
DET_VIEWS_MAX_DT_H = 1.5    # same-site views beyond this are dropped: label fidelity vs the v9
                            # detector is 0.47 at 1 h, 0.39 at 3 h, 0.31 at 6 h (original frame 0.59)
DET_CONF_MIN = 0.05         # v12: 0.05 (RF-DETR scores sit on a different scale than YOLO; v5-v11: 0.1)
DET_NMS_IOU = 0.5
DET_MAX_DET = 100
DET_CONFS = (0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6)   # OOF sweep: per-instance existence cut (the FP/FN knob)
DET_FUSE_IOU = 0.55         # test time: boxes of the fold detectors are fused (WBF) at this IoU
REF_CROP = 256              # refiner input side in native px; boxes whose context exceeds it
REF_CONTEXT = 1.4           #   (REF_CONTEXT x long side) are downscaled, probs upscaled back
REF_PAD = 0.25              # seed channel = detector box grown by this fraction per side
REF_ENCODER = "resnet34"
REF_EPOCHS = 12
REF_BATCH = 16
REF_LR = 3e-4
REF_THRS = (0.6,)           # OOF sweep: refiner mask threshold (v5-v11: flat across 0.5-0.7) ...
REF_MIN_AREAS = (100,)      # ... x minimum instance area (flat 100 vs 200)
REF_NEG_FRAC = 0.0          # v7/v8: 0.25 (hard negatives, WASH both times); 0 = the v6 refiner.
                            # fraction of refiner training samples that are hard negatives -
                            # detector OOF boxes with IoU < REF_NEG_IOU to every GT box of every
                            # record of the file, target all-zero - so the refiner learns to reject
                            # detector FPs (v5/v6 refiner: positives only, 0.95 mean prob on FPs)
REF_NEG_IOU = 0.3
REF_NEG_CONF = 0.2          # only FP boxes above this conf are used (the ones that reach the cut)
REF_OBJ_HEAD = False        # v8: True (objectness head; AUC 0.658 vs detector conf 0.774: WASH)
DUP_FRACTION = 0.5          # a refined mask with more than this fraction of its pixels already
                            # taken by a higher-confidence instance is a duplicate: dropped
if DEBUG:
    DET_EPOCHS, DET_BATCH, REF_EPOCHS = 1, 4, 1
    RFDETR_EPOCHS, RFDETR_RES, RFDETR_BATCH, RFDETR_GRAD_ACCUM, RFDETR_EVAL_INTERVAL = 1, 512, 2, 1, 1
    if not ON_KAGGLE:
        RFDETR_ACCELERATOR = "cpu"
    DET_CONFS, REF_THRS, REF_MIN_AREAS = (0.2, 0.3), (0.5,), (100,)


In [ ]:
# === src/data.py ===
"""COCO loading, polygon rasterisation and grouped folds.

Terminology
-----------
file   : a physical GONG H-alpha observation, e.g. ``20160213153054Bh.jpeg``.
record : one annotator's pass over a file. COCO ``images[].id`` is
         ``<annotator_id>-<file_stem>``; a file can have 1-3 records.
"""
from __future__ import annotations

import json
import random
from collections import defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_util



@lru_cache(maxsize=1)
def data_root() -> Path:
    return find_data_root()


def train_img_dir() -> Path:
    return data_root() / "train" / "train_images"


def test_img_dir() -> Path:
    return data_root() / "test" / "test_images"


def ann_path() -> Path:
    return data_root() / "train" / ANN_NAME


def _as_list(x):
    return json.loads(x) if isinstance(x, str) else x


def load_coco():
    """Return (records, anns_by_record, records_by_file).

    records         : {record_id: image dict}
    anns_by_record  : {record_id: [annotation, ...]}
    records_by_file : {file_name: [record_id, ...]} (sorted)
    """
    d = json.load(open(ann_path()))
    records = {i["id"]: i for i in d["images"]}
    anns_by_record: dict[str, list] = defaultdict(list)
    for a in d["annotations"]:
        anns_by_record[a["image_id"]].append(a)
    records_by_file: dict[str, list[str]] = defaultdict(list)
    for rid, i in records.items():
        records_by_file[i["file_name"]].append(rid)
    for k in records_by_file:
        records_by_file[k].sort()
    return records, dict(anns_by_record), dict(records_by_file)


def annotation_polygons(a) -> list[list[float]]:
    segs = _as_list(a["segmentation"])
    return [s for s in segs if len(s) >= 6]


def annotation_rle(a, h: int = H, w: int = W):
    """Compressed COCO RLE dict for one annotation (full resolution)."""
    polys = annotation_polygons(a)
    if not polys:
        return mask_util.encode(np.asfortranarray(np.zeros((h, w), np.uint8)))
    return mask_util.merge(mask_util.frPyObjects(polys, h, w))


def annotation_mask(a, h: int = H, w: int = W) -> np.ndarray:
    return mask_util.decode(annotation_rle(a, h, w)).astype(np.uint8)


def record_semantic_mask(anns: list, h: int = H, w: int = W) -> np.ndarray:
    m = np.zeros((h, w), np.uint8)
    for a in anns:
        m |= annotation_mask(a, h, w)
    return m


def train_files() -> list[str]:
    return sorted(p.name for p in train_img_dir().glob("*.jpeg"))


def test_files() -> list[str]:
    return sorted(p.name for p in test_img_dir().glob("*.jpeg"))


def make_folds(files: list[str], n_folds: int = 5, seed: int = 2026) -> dict[str, int]:
    """Deterministic grouped folds: the unit is the physical file, so all
    annotator records of one observation land in the same fold."""
    files = sorted(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    return {f: i % n_folds for i, f in enumerate(files)}


In [ ]:
# === src/rle.py ===
"""Submission RLE helpers (pycocotools compressed RLE, 2048x2048)."""
from __future__ import annotations

import numpy as np
from pycocotools import mask as mask_util



def mask_to_rle(mask: np.ndarray) -> str:
    rle = mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def rle_to_mask(counts: str, h: int = H, w: int = W) -> np.ndarray:
    return mask_util.decode({"size": [h, w], "counts": counts.encode("ascii")}).astype(np.uint8)


def masks_to_rles(masks) -> list:
    """Binary masks -> pycocotools RLE dicts (what the metric consumes)."""
    return [mask_util.encode(np.asfortranarray(m.astype(np.uint8))) for m in masks]


In [ ]:
# === src/metric.py ===
"""Panoptic Quality exactly as in the organiser's self-evaluation notebook.

For every annotator record of an image, an (n_gt, n_pred) IoU matrix is
computed. A pair with IoU > 0.5 is a hit; every hit contributes its IoU to
the TP sum. A prediction with no hit is an FP, a GT filament with no hit an
FN. Counts are pooled over *all records of all images*:

    PQ = sum(IoU of hits) / (TP + 0.5 * FP + 0.5 * FN)
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from pycocotools import mask as mask_util

IOU_THRESHOLD = 0.5


def iou_matrix(gt_rles: list, pred_rles: list) -> np.ndarray:
    """gt_rles / pred_rles: lists of pycocotools RLE dicts. Returns (n_gt, n_pred)."""
    if not gt_rles or not pred_rles:
        return np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)
    # iscrowd flags for the *second* argument; all zero -> plain IoU
    m = mask_util.iou(pred_rles, gt_rles, [0] * len(gt_rles))  # (n_pred, n_gt)
    return np.asarray(m, dtype=np.float64).T


@dataclass
class PQAccumulator:
    tp_ious: list[float] = field(default_factory=list)
    fp: int = 0
    fn: int = 0
    n_records: int = 0

    def add(self, gt_rles: list, pred_rles: list) -> dict:
        self.n_records += 1
        n_gt, n_pred = len(gt_rles), len(pred_rles)
        if n_gt == 0:
            self.fp += n_pred
            return {"tp": 0, "fp": n_pred, "fn": 0}
        if n_pred == 0:
            self.fn += n_gt
            return {"tp": 0, "fp": 0, "fn": n_gt}
        m = iou_matrix(gt_rles, pred_rles)
        hit = m > IOU_THRESHOLD
        ious = m[hit].tolist()
        fp = int((hit.sum(0) == 0).sum())
        fn = int((hit.sum(1) == 0).sum())
        self.tp_ious.extend(ious)
        self.fp += fp
        self.fn += fn
        return {"tp": len(ious), "fp": fp, "fn": fn, "sum_iou": float(sum(ious))}

    @property
    def tp(self) -> int:
        return len(self.tp_ious)

    def summary(self) -> dict:
        denom = self.tp + 0.5 * self.fp + 0.5 * self.fn
        sum_iou = float(sum(self.tp_ious))
        pq = sum_iou / denom if denom > 0 else 0.0
        sq = sum_iou / self.tp if self.tp else 0.0
        rq = self.tp / denom if denom > 0 else 0.0
        return {
            "pq": pq, "sq": sq, "rq": rq,
            "tp": self.tp, "fp": self.fp, "fn": self.fn, "records": self.n_records,
        }


def score_predictions(gt_by_record: dict[str, list], preds_by_file: dict[str, list],
                      records_by_file: dict[str, list[str]], files: list[str]) -> dict:
    """gt_by_record: {record_id: [rle dict]}, preds_by_file: {file_stem: [rle dict]}."""
    acc = PQAccumulator()
    for f in files:
        stem = f.rsplit(".", 1)[0]
        preds = preds_by_file.get(stem, [])
        for rid in records_by_file[f]:
            acc.add(gt_by_record[rid], preds)
    return acc.summary()


In [ ]:
# === src/postprocess.py ===
"""Turn a probability map into filament instances at native resolution."""
from __future__ import annotations

import cv2
import numpy as np



def upsample(prob: np.ndarray) -> np.ndarray:
    """Any-size prob map -> float32 (H, W)."""
    if prob.dtype == np.uint8:
        prob = prob.astype(np.float32) / 255.0
    if prob.shape != (H, W):
        prob = cv2.resize(prob.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    return prob.astype(np.float32, copy=False)


def probs_to_instances(prob: np.ndarray, threshold: float = 0.5, min_area: int = 150,
                       close_kernel: int = 0, min_score: float = 0.0,
                       fill_holes: bool = True, dilate: int = 0) -> list[np.ndarray]:
    """Threshold -> optional morphological closing -> 8-connected components
    -> drop components smaller than `min_area` or whose mean probability is
    below `min_score` (a confidence filter on top of the size filter)
    -> optionally grow (dilate > 0) or shrink (dilate < 0) each kept component
    by |dilate| px (boundary-bias probe; applied per component so neighbours
    are not merged).

    Returns a list of uint8 (H, W) masks, largest first. Closing bridges tiny
    gaps so one filament is less likely to be emitted as several fragments
    (fragments fail the IoU > 0.5 match and each costs 0.5 FP).
    """
    prob = upsample(prob)
    binary = (prob >= threshold).astype(np.uint8)
    if close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    out = []
    order = np.argsort(-stats[1:, cv2.CC_STAT_AREA]) + 1
    for lab in order:
        if stats[lab, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y, w, h = stats[lab, :4]
        sub = (labels[y:y + h, x:x + w] == lab).astype(np.uint8)
        if min_score > 0 and float(prob[y:y + h, x:x + w][sub > 0].mean()) < min_score:
            continue
        if fill_holes:
            contours, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(sub, contours, -1, 1, thickness=cv2.FILLED)
            # a hole may contain ANOTHER component; never paint over it (instances must
            # not overlap - the evaluator errors on a single shared pixel)
            others = labels[y:y + h, x:x + w]
            sub[(others != 0) & (others != lab)] = 0
        m = np.zeros(prob.shape, np.uint8)
        if dilate:
            k = abs(dilate)
            y0, x0 = max(y - k, 0), max(x - k, 0)
            y1, x1 = min(y + h + k, prob.shape[0]), min(x + w + k, prob.shape[1])
            pad = np.zeros((y1 - y0, x1 - x0), np.uint8)
            pad[y - y0:y - y0 + h, x - x0:x - x0 + w] = sub
            se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1))
            grown = cv2.dilate(pad, se) if dilate > 0 else cv2.erode(pad, se)
            if grown.any():
                m[y0:y1, x0:x1] = grown
                out.append(m)
                continue
        m[y:y + h, x:x + w] = sub
        out.append(m)
    return resolve_overlaps(out)


def resolve_overlaps(masks: list[np.ndarray]) -> list[np.ndarray]:
    """Hard guarantee that no two instances share a pixel (the evaluator errors on one
    shared pixel). Earlier (larger) instances win; an instance emptied by clipping is dropped."""
    if len(masks) < 2:
        return masks
    taken = np.zeros(masks[0].shape, bool)
    out = []
    for m in masks:
        b = m.astype(bool) & ~taken
        if b.any():
            taken |= b
            out.append(b.astype(np.uint8))
    return out


def instances_never_empty(prob: np.ndarray, threshold: float, min_area: int,
                          close_kernel: int = 0, min_score: float = 0.0) -> list[np.ndarray]:
    """Like probs_to_instances but guarantees at least one instance so every
    test image has a submission row."""
    inst = probs_to_instances(prob, threshold, min_area, close_kernel, min_score)
    if not inst:
        inst = probs_to_instances(prob, threshold * 0.5, max(min_area // 3, 10), close_kernel)[:1]
    if not inst:
        p = upsample(prob)
        m = np.zeros(p.shape, np.uint8)
        y, x = np.unravel_index(int(p.argmax()), p.shape)
        m[y:y + 4, x:x + 4] = 1
        inst = [m]
    return inst


In [ ]:
# === src/dataset.py ===
"""Cache building and torch datasets over the downscaled cache.

cache/img{S}/<stem>.png        grayscale image downscaled to S x S
cache/test{S}/<stem>.png       same for test images
cache/mask{S}/<record_id>.png  union mask of one annotator record at S x S
cache/gt_rles.pkl              {record_id: [compressed RLE dict, ...]} at 2048 (for PQ)
cache/meta.json                size, records_by_file, folds
"""
from __future__ import annotations

import json
import pickle
import random

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset



def build_cache(size: int = IMG_SIZE, n_folds: int = N_FOLDS) -> dict:
    """Idempotent: skips files that already exist. Returns meta."""
    records, anns, by_file = load_coco()
    img_dir, tst_dir, msk_dir = CACHE / f"img{size}", CACHE / f"test{size}", CACHE / f"mask{size}"
    for d in (img_dir, tst_dir, msk_dir):
        d.mkdir(parents=True, exist_ok=True)
    gt_rles = {}
    for n, f in enumerate(train_files()):
        stem = f.rsplit(".", 1)[0]
        out = img_dir / f"{stem}.png"
        if not out.exists():
            im = cv2.imread(str(train_img_dir() / f), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(out), cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA))
        for rid in by_file[f]:
            a = anns.get(rid, [])
            gt_rles[rid] = [annotation_rle(x) for x in a if not int(x.get("iscrowd", 0))]
            mout = msk_dir / f"{rid}.png"
            if not mout.exists():
                m = record_semantic_mask(a).astype(np.float32)
                m = cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
                cv2.imwrite(str(mout), ((m >= 0.5) * 255).astype(np.uint8))
        if n % 100 == 0:
            log(f"cache train {n}")
    for f in test_files():
        out = tst_dir / (f.rsplit(".", 1)[0] + ".png")
        if not out.exists():
            im = cv2.imread(str(test_img_dir() / f), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(out), cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA))
    with open(CACHE / "gt_rles.pkl", "wb") as fh:
        pickle.dump(gt_rles, fh)
    meta = {"size": size, "records_by_file": by_file, "folds": make_folds(train_files(), n_folds, SEED),
            "medoid": medoid_records(by_file, msk_dir)}
    json.dump(meta, open(CACHE / "meta.json", "w"))
    log(f"cache ready: {len(by_file)} files, {len(gt_rles)} records, {len(test_files())} test")
    return meta


def medoid_records(by_file: dict, msk_dir) -> dict:
    """{file: record_id} — for multi-annotator files, the record whose union mask has
    the highest mean IoU with the other records' masks (the consensus annotator).
    Single-record files map to their only record."""
    out = {}
    for f, rids in by_file.items():
        if len(rids) == 1:
            out[f] = rids[0]
            continue
        masks = [_read(msk_dir / f"{r}.png") > 127 for r in rids]
        best, best_score = rids[0], -1.0
        for i, r in enumerate(rids):
            ious = []
            for j, m in enumerate(masks):
                if j == i:
                    continue
                u = (masks[i] | m).sum()
                ious.append((masks[i] & m).sum() / u if u else 1.0)
            s = float(np.mean(ious))
            if s > best_score:
                best, best_score = r, s
        out[f] = best
    return out


def load_meta() -> dict:
    return json.load(open(CACHE / "meta.json"))


def load_gt_rles() -> dict:
    return pickle.load(open(CACHE / "gt_rles.pkl", "rb"))


def _read(path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise FileNotFoundError(path)
    return im


def normalise(img: np.ndarray) -> np.ndarray:
    return (img.astype(np.float32) / 255.0 - 0.5) / 0.25


class CropDataset(Dataset):
    """Random crops from cached images. Each __getitem__ picks one annotator
    record at random for the file (label augmentation across annotators)."""

    def __init__(self, files: list[str], meta: dict, crop: int, samples_per_file: int,
                 positive_bias: float = POSITIVE_BIAS, target_mode: str = TARGET_MODE):
        self.files, self.meta = files, meta
        self.S, self.crop, self.spf, self.pos = meta["size"], crop, samples_per_file, positive_bias
        self.img_dir, self.msk_dir = CACHE / f"img{self.S}", CACHE / f"mask{self.S}"
        self.target_mode = target_mode

    def __len__(self):
        return len(self.files) * self.spf

    def __getitem__(self, i):
        f = self.files[i // self.spf]
        stem = f.rsplit(".", 1)[0]
        if self.target_mode == "medoid":
            rid = self.meta["medoid"][f]
        else:
            rid = random.choice(self.meta["records_by_file"][f])
        img = _read(self.img_dir / f"{stem}.png")
        msk = _read(self.msk_dir / f"{rid}.png") > 127

        c = self.crop
        if random.random() < self.pos and msk.any():
            ys, xs = np.nonzero(msk)
            k = random.randrange(len(ys))
            cy, cx = ys[k] + random.randint(-c // 3, c // 3), xs[k] + random.randint(-c // 3, c // 3)
            y0 = int(np.clip(cy - c // 2, 0, self.S - c))
            x0 = int(np.clip(cx - c // 2, 0, self.S - c))
        else:
            y0, x0 = random.randint(0, self.S - c), random.randint(0, self.S - c)
        img, msk = img[y0:y0 + c, x0:x0 + c], msk[y0:y0 + c, x0:x0 + c]

        # geometric augs: filaments have no canonical orientation on the disk
        if random.random() < 0.5:
            img, msk = img[:, ::-1], msk[:, ::-1]
        if random.random() < 0.5:
            img, msk = img[::-1], msk[::-1]
        k = random.randrange(4)
        img, msk = np.rot90(img, k), np.rot90(msk, k)

        # photometric augs: GONG sites differ in exposure/contrast
        x = img.astype(np.float32) / 255.0
        if random.random() < 0.8:
            x = np.clip(x, 0, 1) ** random.uniform(0.7, 1.4)
        if random.random() < 0.8:
            x = x * random.uniform(0.8, 1.2) + random.uniform(-0.08, 0.08)
        if random.random() < 0.3:
            x = x + np.random.normal(0, random.uniform(0.005, 0.03), x.shape).astype(np.float32)
        x = (np.clip(x, 0, 1) - 0.5) / 0.25
        return (torch.from_numpy(np.ascontiguousarray(x))[None],
                torch.from_numpy(np.ascontiguousarray(msk).astype(np.float32))[None])


class FullImageDataset(Dataset):
    """Whole cached images (S x S) for validation / test inference."""

    def __init__(self, files: list[str], size: int, split: str = "train"):
        self.files = files
        self.dir = CACHE / (f"img{size}" if split == "train" else f"test{size}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        f = self.files[i]
        img = _read(self.dir / (f.rsplit(".", 1)[0] + ".png"))
        return torch.from_numpy(normalise(img))[None], f


In [ ]:
# === src/model.py ===
"""Network, loss, device and TTA forward."""
from __future__ import annotations

import os

import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return torch.device("mps")
    return torch.device("cpu")


def build_model(arch: str = "unet", encoder: str = "resnet34", pretrained: bool = True):
    cls = {"unet": smp.Unet, "unetpp": smp.UnetPlusPlus, "fpn": smp.FPN,
           "deeplab": smp.DeepLabV3Plus, "manet": smp.MAnet}[arch]
    return cls(encoder_name=encoder, encoder_weights="imagenet" if pretrained else None,
               in_channels=1, classes=1)


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor, dice_weight: float = 1.0,
                  pos_weight: float = 1.0) -> torch.Tensor:
    pw = torch.tensor(pos_weight, device=logits.device)
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + 1) / (denom + 1)
    return bce + dice_weight * dice.mean()


@torch.no_grad()
def predict_probs(model, x: torch.Tensor, tta: bool = True) -> torch.Tensor:
    """x: (B,1,S,S) -> sigmoid probs (B,1,S,S), optional 4x flip TTA."""
    out = torch.sigmoid(model(x))
    if not tta:
        return out
    out = out + torch.sigmoid(model(x.flip(-1))).flip(-1)
    out = out + torch.sigmoid(model(x.flip(-2))).flip(-2)
    out = out + torch.sigmoid(model(x.flip(-1, -2))).flip(-1, -2)
    return out / 4


def save_checkpoint(path, model, **meta):
    torch.save({"state_dict": model.state_dict(), **meta}, path)


def load_checkpoint(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    model = build_model(ck["arch"], ck["encoder"], pretrained=False).to(device)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    return model, ck


In [ ]:
# === src/evaluate.py ===
"""Model -> probability maps -> instances -> Panoptic Quality."""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import DataLoader



@torch.no_grad()
def predict_files(model, files, size, device, split="train", tta=True, batch_size=None):
    """Yield (file_name, prob map float32 at model resolution (size x size))."""
    model.eval()
    if batch_size is None:
        batch_size = 2 if size <= 1024 else 1   # a 2048^2 U-Net forward is ~4x the activations
    dl = DataLoader(FullImageDataset(files, size, split), batch_size=batch_size,
                    num_workers=min(WORKERS, 2), shuffle=False)
    for x, names in dl:
        p = predict_probs(model, x.to(device), tta=tta).float().cpu().numpy()[:, 0]
        for prob, name in zip(p, names):
            yield name, prob


def score_probs(probs: dict, meta: dict, gt_rles: dict, threshold: float, min_area: int,
                close_kernel: int = 0, min_score: float = 0.0, dilate: int = 0) -> dict:
    """probs: {file_name: prob map (any size, float or uint8)} -> PQ summary."""
    acc = PQAccumulator()
    for name, prob in probs.items():
        preds = masks_to_rles(probs_to_instances(prob, threshold, min_area, close_kernel, min_score,
                                                 dilate=dilate))
        for rid in meta["records_by_file"][name]:
            acc.add(gt_rles[rid], preds)
    return acc.summary()


def evaluate(model, files, meta, device, gt_rles, thresholds=(0.5,), min_area=150, tta=False,
             keep_probs=False):
    """Run the model on `files` and score PQ at each threshold.
    Returns ({thr: summary}, {file: uint8 prob map at model size} if keep_probs else None)."""
    accs = {t: PQAccumulator() for t in thresholds}
    kept = {} if keep_probs else None
    for name, prob in predict_files(model, files, meta["size"], device, tta=tta):
        if keep_probs:
            kept[name] = np.round(prob * 255).astype(np.uint8)
        for t in thresholds:
            preds = masks_to_rles(probs_to_instances(prob, t, min_area))
            for rid in meta["records_by_file"][name]:
                accs[t].add(gt_rles[rid], preds)
    return {t: a.summary() for t, a in accs.items()}, kept


In [ ]:
# === src/views.py ===
"""Training-time views from the public GONG archive (host ruling: training only).

An archive frame is NOT the competition's processing. Measured on 24 competition/archive
copies of the SAME frames (2026-09-28): no shift (< 0.1 px), no rotation (< 0.005 deg), no
scale (radius ratio 1.0015 +- 0.002, i.e. disk-fit noise), but the organisers clipped the
highlights (a monotone LUT: identity to ~150 then flat at ~152; archive p99 194 -> 156)
and their JPEGs carry ~1.5-2x more high-frequency content that the archive JPEGs do not
contain (no unsharp setting recovers it: it is a less-compressed source, not a filter).
A view is therefore (1) the archive frame through the LUT - no geometric warp - with a
residual mild blur gap that YOLO's 2048 -> 1280 downscale and blur augmentation cover,
and (2) labelled by projecting the competition boxes through solar differential rotation
for the elapsed time (identity for other-site frames of the same minute).
"""
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import cv2
import numpy as np


# Snodgrass (1990) synodic differential rotation, deg/day
def omega_synodic_deg_per_day(lat: np.ndarray) -> np.ndarray:
    s2 = np.sin(lat) ** 2
    return 14.713 - 2.396 * s2 - 1.787 * s2 * s2 - 0.9856


def disk_fit(img: np.ndarray) -> tuple[float, float, float]:
    """(cx, cy, R) of the solar disk: Otsu on a 4x-downsampled frame, largest blob."""
    small = cv2.resize(img, (img.shape[1] // 4, img.shape[0] // 4), interpolation=cv2.INTER_AREA)
    _, b = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, cents = cv2.connectedComponentsWithStats(b, connectivity=8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    cx, cy = cents[k] * 4 + 1.5
    return float(cx), float(cy), float(np.sqrt(stats[k, cv2.CC_STAT_AREA] / np.pi) * 4)


def align_geometry(view: np.ndarray, disk_view, disk_ref) -> np.ndarray:
    """Similarity transform: view disk (cx, cy, R) -> reference disk. Output HxW."""
    cxv, cyv, Rv = disk_view
    cxr, cyr, Rr = disk_ref
    s = Rr / Rv
    M = np.array([[s, 0, cxr - s * cxv], [0, s, cyr - s * cyv]], np.float32)
    return cv2.warpAffine(view, M, (W, H), flags=cv2.INTER_LINEAR, borderValue=0)


def intensity_lut(pairs: list[tuple[np.ndarray, np.ndarray]]) -> np.ndarray:
    """Monotone 256-entry LUT archive -> competition from (competition, aligned archive)
    frame pairs: for each archive grey level, the median competition grey level over all
    pixels of that level (on-disk and background alike), smoothed and made monotone."""
    sums = np.zeros(256); cnts = np.zeros(256)
    for comp, arch in pairs:
        a = arch.ravel(); c = comp.ravel().astype(np.float64)
        sums += np.bincount(a, weights=c, minlength=256); cnts += np.bincount(a, minlength=256)
    lut = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan)
    idx = np.arange(256)
    ok = ~np.isnan(lut)
    lut = np.interp(idx, idx[ok], lut[ok])
    lut = np.maximum.accumulate(lut)               # monotone
    return np.clip(np.round(lut), 0, 255).astype(np.uint8)


def t_of(stem: str) -> datetime:
    return datetime.strptime(stem[:14], "%Y%m%d%H%M%S")


def project_boxes(boxes: np.ndarray, disk_ref, dt_days: float) -> tuple[np.ndarray, np.ndarray]:
    """Boxes (n, 4) xywh in the reference frame -> xywh in the same geometry after dt_days of
    rotation (the view has already been aligned to the reference disk). valid=False for
    boxes that leave the usable disk."""
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32), np.zeros(0, bool)
    cx, cy, R = disk_ref
    x = boxes[:, 0] + boxes[:, 2] / 2; y = boxes[:, 1] + boxes[:, 3] / 2
    X = (x - cx) / R; Y = (cy - y) / R
    valid = X * X + Y * Y < 0.94
    lat = np.arcsin(np.clip(Y, -1, 1)); coslat = np.maximum(np.cos(lat), 1e-6)
    lon = np.arcsin(np.clip(X / coslat, -1, 1))
    lon2 = lon + np.deg2rad(omega_synodic_deg_per_day(lat) * dt_days)
    valid &= np.abs(lon2) < np.deg2rad(75)
    fore = np.maximum(np.cos(lon2), 1e-3) / np.maximum(np.cos(lon), 1e-3)
    x2 = cx + coslat * np.sin(lon2) * R
    w2 = boxes[:, 2] * fore; h2 = boxes[:, 3]
    out = np.stack([x2 - w2 / 2, y - h2 / 2, w2, h2], 1).astype(np.float32)
    return out, valid


COMPETITION_SITES = "BCLMTU"   # archive sites S and Z are not in the competition and sit at
                               # different roll angles (up to 32 deg measured) - never used


def ecc_align(ref: np.ndarray, view: np.ndarray, max_roll_deg: float = 1.0, min_corr: float = 0.9):
    """Rigid (roll + shift) alignment of a same-minute OTHER-SITE frame onto the reference
    frame, estimated by ECC on 512 px copies (sites differ by 2-15 px of centring and
    < 0.5 deg of roll). Returns (warped view at full res, info dict) or (None, info) when
    the estimate is not trustworthy. Same-site frames must NOT go through this (they are
    already in the reference geometry; their labels are rotated instead)."""
    cs = cv2.resize(ref.astype(np.float32), (512, 512), interpolation=cv2.INTER_AREA)
    vs = cv2.resize(view.astype(np.float32), (512, 512), interpolation=cv2.INTER_AREA)
    warp = np.eye(2, 3, dtype=np.float32)
    try:
        cc, warp = cv2.findTransformECC(cs, vs, warp, cv2.MOTION_EUCLIDEAN,
                                        (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 300, 1e-6), None, 7)
    except cv2.error:
        return None, {"ok": False, "reason": "ecc failed"}
    roll = float(np.degrees(np.arctan2(warp[1, 0], warp[0, 0])))
    info = {"ok": True, "roll": roll, "dx": float(warp[0, 2] * 4), "dy": float(warp[1, 2] * 4), "corr": float(cc)}
    if abs(roll) > max_roll_deg or cc < min_corr:
        info["ok"] = False; info["reason"] = "roll/corr out of range"
        return None, info
    full = warp.copy(); full[:, 2] *= 4
    # warp maps ref -> view; WARP_INVERSE_MAP brings the view into the ref frame
    out = cv2.warpAffine(view, full, (W, H), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP, borderValue=0)
    return out, info


DEFAULT_LUT = None


def default_lut() -> np.ndarray:
    """The measured archive -> competition LUT (identity up to ~150, clipped above)."""
    global DEFAULT_LUT
    if DEFAULT_LUT is None:
        idx = np.arange(256, dtype=np.float32)
        DEFAULT_LUT = np.clip(np.minimum(idx * 0.995, 152), 0, 255).astype(np.uint8)
    return DEFAULT_LUT


def make_view(view_img: np.ndarray, lut: np.ndarray | None = None) -> np.ndarray:
    """Competition-processed view of an archive frame: LUT only (no geometric warp - none is
    needed, see module docstring)."""
    return (lut if lut is not None else default_lut())[view_img]


In [ ]:
# === src/detector.py ===
"""Rung 4, stage 1: YOLO box detector (ultralytics) trained on the 1024 px cache.

One box per filament is the whole point: NMS gives instance separation (no merged
neighbours) and a per-object confidence (the FP/FN knob), which the semantic model
never had. Labels are written once (YOLO reads static txt files, so one annotator
record per file: the consensus / medoid one); per-fold train/val lists reuse the
grouped folds of the cache. Boxes are returned in NATIVE (2048) coordinates.
"""
from __future__ import annotations

import os
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from pycocotools import mask as mask_util



def yolo_root() -> Path:
    return CACHE / ("yolo" if DET_SOURCE == "cache" else "yolo_native")


LABEL_BOXES: dict = {}   # {train file: [(x, y, w, h) native px]} - filled by build_yolo_labels


# --- RF-DETR (v12): same labels / folds / outputs as the YOLO path --------------------------
def _rfdetr_cls():
    import rfdetr
    return {"nano": rfdetr.RFDETRNano, "small": rfdetr.RFDETRSmall, "medium": rfdetr.RFDETRMedium,
            "base": rfdetr.RFDETRBase}[RFDETR_SIZE]


def rfdetr_fold_dataset(fold: int, meta: dict, train_limit: int = 0, val_limit: int = 0) -> Path:
    """Roboflow/COCO layout RF-DETR expects: <root>/{train,valid,test}/_annotations.coco.json with
    the images beside it (symlinks to the native 2048 JPEGs). One category, id 0, 'filament'.
    test = the fold's val files (RF-DETR wants the split; run_test stays off)."""
    import json
    root = CACHE / "rfdetr" / f"fold{fold}"
    shutil.rmtree(root, ignore_errors=True)
    folds = meta["folds"]
    tr = sorted(f for f, k in folds.items() if k != fold)
    va = sorted(f for f, k in folds.items() if k == fold)
    if train_limit:
        tr = tr[:train_limit]
    if val_limit:
        va = va[:val_limit]
    n_ann = 0
    for split, fs in (("train", tr), ("valid", va), ("test", va)):
        d = root / split
        d.mkdir(parents=True)
        images, anns = [], []
        for i, f in enumerate(fs):
            os.symlink((train_img_dir() / f).resolve(), d / f)
            images.append({"id": i, "file_name": f, "width": W, "height": H})
            for x, y, w, h in LABEL_BOXES.get(f, []):
                anns.append({"id": len(anns), "image_id": i, "category_id": 0,
                             "bbox": [float(x), float(y), float(w), float(h)], "area": float(w * h), "iscrowd": 0})
        json.dump({"images": images, "annotations": anns,
                   "categories": [{"id": 0, "name": "filament", "supercategory": "none"}]},
                  open(d / "_annotations.coco.json", "w"))
        if split == "train":
            n_ann = len(anns)
    log(f"fold {fold}: RF-DETR dataset {len(tr)} train files ({n_ann} boxes) / {len(va)} val files")
    return root


def _train_rfdetr(fold: int, meta: dict, out_dir: Path, train_limit: int = 0, val_limit: int = 0) -> Path:
    ds = rfdetr_fold_dataset(fold, meta, train_limit, val_limit)
    od = out_dir / "rfdetr" / f"fold{fold}"
    log(f"fold {fold}: RF-DETR {RFDETR_SIZE} res {RFDETR_RES} epochs {RFDETR_EPOCHS} batch "
        f"{RFDETR_BATCH}x{RFDETR_GRAD_ACCUM} lr {RFDETR_LR}")
    model = _rfdetr_cls()(resolution=RFDETR_RES)
    model.train(dataset_dir=str(ds), epochs=RFDETR_EPOCHS, batch_size=RFDETR_BATCH,
                grad_accum_steps=RFDETR_GRAD_ACCUM, lr=RFDETR_LR, output_dir=str(od), tensorboard=False,
                num_workers=WORKERS, eval_interval=RFDETR_EVAL_INTERVAL, seed=SEED,
                accelerator=RFDETR_ACCELERATOR, expanded_scales=RFDETR_EXPANDED_SCALES)
    best = next((od / n for n in ("checkpoint_best_total.pth", "checkpoint_best_ema.pth",
                                  "checkpoint_best_regular.pth", "last_ema.pth", "checkpoint.pth")
                 if (od / n).exists()), None)
    if best is None:
        raise FileNotFoundError(f"no RF-DETR checkpoint in {od}: {sorted(p.name for p in od.glob('*'))}")
    dst = out_dir / f"fold{fold}_rfdetr.pth"
    shutil.copy2(best, dst)
    shutil.rmtree(od, ignore_errors=True)
    return dst


def _load_rfdetr(weights):
    cls = _rfdetr_cls()
    try:
        return cls.from_checkpoint(str(weights))
    except Exception:  # noqa: BLE001 - older API: weights through the constructor
        return cls(pretrain_weights=str(weights), resolution=RFDETR_RES)


def _predict_rfdetr(weights, files: list[str], split: str, conf: float, model=None) -> dict:
    from PIL import Image
    m = model if model is not None else _load_rfdetr(weights)
    d = train_img_dir() if split == "train" else test_img_dir()
    out = {}
    for i in range(0, len(files), 8):
        batch = files[i:i + 8]
        imgs = [Image.open(d / f).convert("RGB") for f in batch]
        dets = m.predict(imgs, threshold=conf)
        if not isinstance(dets, list):
            dets = [dets]
        for f, det in zip(batch, dets):
            if det is None or len(det) == 0:
                out[f] = np.zeros((0, 5), np.float32)
                continue
            b = np.concatenate([np.asarray(det.xyxy, np.float32),
                                np.asarray(det.confidence, np.float32)[:, None]], 1)
            out[f] = b[np.argsort(-b[:, 4])][:DET_MAX_DET]
    return out


VIEWS_DATASET = "gong-halpha-train-views-long"


def views_of(stem: str) -> list[Path]:
    """Archive frames around a train file (training-time extra views), wherever they live:
    data/sequences_long/<stem>/ locally, or the attached Kaggle dataset (auto-unzipped per stem)."""
    roots = [REPO / "data" / "sequences_long"]
    kin = Path("/kaggle/input")
    if kin.is_dir():
        roots += [p for p in kin.glob(f"*/{VIEWS_DATASET}")] + [p for p in kin.glob(f"*/*/{VIEWS_DATASET}")] + \
                 [kin / VIEWS_DATASET]
    for r in roots:
        for d in (r / stem, r / stem / stem):
            if d.is_dir():
                return sorted(p for p in d.glob("*.jpg") if p.stem != stem)
    return []


def build_views(stem: str, ref_img: np.ndarray, boxes_xywh: np.ndarray, max_views: int):
    """Yield (view_name, view_image uint8 HxW, boxes xywh) for the usable archive frames of a
    train file: other competition sites at the same minute (ECC-aligned onto the reference,
    labels unchanged) and the same site within DET_VIEWS_MAX_DT_H (labels rotated)."""
    disk_ref = None
    n = 0
    for p in views_of(stem):
        if n >= max_views or p.stem[14] not in COMPETITION_SITES:
            continue
        dt_days = (t_of(p.stem) - t_of(stem)).total_seconds() / 86400
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None or img.shape != (H, W):
            continue
        v = make_view(img)
        if p.stem[14] != stem[14]:
            if abs(dt_days) * 24 > 0.2:
                continue
            v, info = ecc_align(ref_img, v)
            if v is None:
                continue
            bx = boxes_xywh.copy()
        else:
            if abs(dt_days) * 24 > DET_VIEWS_MAX_DT_H:
                continue
            disk_ref = disk_ref or disk_fit(ref_img)
            bx, valid = project_boxes(boxes_xywh, disk_ref, dt_days)
            bx = bx[valid]
        n += 1
        yield p.stem, v, bx


def image_source(f: str, size: int, split: str = "train") -> Path:
    """The image the detector trains on / predicts from for file `f`."""
    if DET_SOURCE == "native":
        return (train_img_dir() if split == "train" else test_img_dir()) / f
    return CACHE / (f"img{size}" if split == "train" else f"test{size}") / (f.rsplit(".", 1)[0] + ".png")


def box_of_rle(rle) -> tuple[float, float, float, float]:
    """(x, y, w, h) in native px of one GT RLE."""
    x, y, w, h = mask_util.toBbox(rle)
    return float(x), float(y), float(w), float(h)


def union_boxes(rles_by_record: list[list], min_support: float = DET_LABEL_MIN_SUPPORT,
                iou_thr: float = 0.5) -> list[tuple[float, float, float, float]]:
    """Filaments of all annotator records of one file, deduplicated across records.
    Greedy clustering: a filament joins the first cluster that holds no mask of its own record
    yet and holds a mask of another record with IoU > iou_thr (one filament per record per
    cluster, so an annotator who split a filament in two keeps two boxes). A cluster's support
    is the number of records in it; clusters with support / n_records >= min_support are kept,
    each represented by the box of its member with the highest total IoU to the other members
    (the consensus outline)."""
    n = len(rles_by_record)
    clusters: list[list[tuple[int, object]]] = []   # [(record index, rle), ...]
    for r, rles in enumerate(rles_by_record):
        for rle in rles:
            placed = False
            for cl in clusters:
                if any(k == r for k, _ in cl):
                    continue
                others = [m for _, m in cl]
                if (np.asarray(mask_util.iou(others, [rle], [0] * 1)) > iou_thr).any():
                    cl.append((r, rle)); placed = True
                    break
            if not placed:
                clusters.append([(r, rle)])
    out = []
    for cl in clusters:
        if len({k for k, _ in cl}) / n < min_support:
            continue
        if len(cl) == 1:
            out.append(box_of_rle(cl[0][1])); continue
        ms = [m for _, m in cl]
        tot = np.asarray(mask_util.iou(ms, ms, [0] * len(ms))).sum(1)
        out.append(box_of_rle(ms[int(tot.argmax())]))
    return out


def build_yolo_labels(meta: dict, gt_rles: dict) -> Path:
    """images/<stem>.<ext> -> symlinks to the DET_SOURCE images (cached S x S PNGs or the native
    JPEGs); labels/<stem>.txt in normalised YOLO format. Label files are always rewritten
    (the label recipe is a config knob)."""
    root, S = yolo_root(), meta["size"]
    (root / "images").mkdir(parents=True, exist_ok=True)
    (root / "labels").mkdir(exist_ok=True)
    (root / "labels.cache").unlink(missing_ok=True)   # ultralytics' scan cache of the old labels
    n_box, n_files, n_views, support_hist = 0, 0, 0, {}
    for f in train_files():
        stem = f.rsplit(".", 1)[0]
        src = image_source(f, S)
        dst = root / "images" / (stem + src.suffix)
        if not dst.exists():
            os.symlink(src.resolve(), dst)
        rids = meta["records_by_file"][f]
        if DET_LABEL_RECORD == "union":
            boxes = union_boxes([gt_rles[r] for r in rids])
        else:
            rid = meta["medoid"][f] if DET_LABEL_RECORD == "medoid" else rids[0]
            boxes = [box_of_rle(rle) for rle in gt_rles[rid]]
        LABEL_BOXES[f] = [(x, y, w, h) for x, y, w, h in boxes if w >= 1 and h >= 1]   # native xywh, reused by RF-DETR
        lines = []
        for x, y, w, h in boxes:
            if w < 1 or h < 1:
                continue
            lines.append(f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
        label_txt = "\n".join(lines) + ("\n" if lines else "")
        (root / "labels" / f"{stem}.txt").write_text(label_txt)
        n_box += len(lines); n_files += 1
        support_hist[len(rids)] = support_hist.get(len(rids), 0) + len(lines)
        if DET_EXTRA_VIEWS > 0:
            # extra training views from the archive (src/views.py): materialised as JPEGs with
            # their own (aligned / rotated) box labels; masks are never reused
            ref_img = cv2.imread(str(train_img_dir() / f), cv2.IMREAD_GRAYSCALE)
            bxy = np.array([[x, y, w, h] for x, y, w, h in boxes if w >= 1 and h >= 1], np.float32).reshape(-1, 4)
            for vname, vimg, vboxes in build_views(stem, ref_img, bxy, DET_EXTRA_VIEWS):
                vdst = root / "images" / f"{stem}__{vname}.jpg"
                if not vdst.exists():
                    cv2.imwrite(str(vdst), vimg, [cv2.IMWRITE_JPEG_QUALITY, 95])
                vl = [f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}"
                      for x, y, w, h in vboxes if w >= 1 and h >= 1]
                (root / "labels" / f"{stem}__{vname}.txt").write_text("\n".join(vl) + ("\n" if vl else ""))
                n_views += 1
    log(f"YOLO labels ({DET_LABEL_RECORD}): {n_box} boxes over {n_files} files "
        f"({n_box / max(n_files, 1):.2f}/file); boxes by n_annotators {support_hist}"
        + (f"; extra views {n_views} ({n_views / max(n_files, 1):.1f}/file)" if DET_EXTRA_VIEWS else ""))
    return root


def fold_yaml(fold: int, meta: dict, train_limit: int = 0, val_limit: int = 0) -> Path:
    root, folds = yolo_root(), meta["folds"]
    tr = sorted(f for f, k in folds.items() if k != fold)
    va = sorted(f for f, k in folds.items() if k == fold)
    if train_limit:
        tr = tr[:train_limit]
    if val_limit:
        va = va[:val_limit]
    ext = image_source(next(iter(folds)), meta["size"]).suffix
    paths = lambda fs: "\n".join(str(root / "images" / (f.rsplit(".", 1)[0] + ext)) for f in fs) + "\n"
    train_txt = paths(tr)
    if DET_EXTRA_VIEWS > 0:   # views of TRAIN-fold files only; validation stays the labelled frames
        extra = [p for f in tr for p in sorted((root / "images").glob(f"{f.rsplit('.', 1)[0]}__*{ext}"))]
        train_txt += "\n".join(str(p) for p in extra) + ("\n" if extra else "")
    (root / f"fold{fold}_train.txt").write_text(train_txt)
    (root / f"fold{fold}_val.txt").write_text(paths(va))
    y = root / f"fold{fold}.yaml"
    y.write_text(f"path: {root}\ntrain: {root / f'fold{fold}_train.txt'}\nval: {root / f'fold{fold}_val.txt'}\n"
                 f"names:\n  0: filament\n")
    return y


def _device_arg():
    return 0 if torch.cuda.is_available() else "cpu"


def train_detector(fold: int, meta: dict, out_dir: Path, epochs: int = DET_EPOCHS, train_limit: int = 0,
                   val_limit: int = 0) -> Path:
    """Train the fold's detector: YOLO -> fold{k}_yolo.pt, or RF-DETR (DET_ARCH) -> fold{k}_rfdetr.pth."""
    if DET_ARCH == "rfdetr":
        return _train_rfdetr(fold, meta, out_dir, train_limit, val_limit)
    from ultralytics import YOLO
    from ultralytics import settings as yolo_settings
    yolo_settings.update({"sync": False, "wandb": False, "mlflow": False, "tensorboard": False})
    yaml = fold_yaml(fold, meta, train_limit, val_limit)
    log(f"fold {fold}: YOLO {DET_MODEL} imgsz {DET_IMGSZ} epochs {epochs}")
    model = YOLO(DET_MODEL)
    model.train(data=str(yaml), epochs=epochs, imgsz=DET_IMGSZ, batch=DET_BATCH, workers=WORKERS,
                seed=SEED, deterministic=False, project=str(out_dir / "yolo"), name=f"fold{fold}",
                exist_ok=True, plots=False, verbose=False, device=_device_arg(), single_cls=True,
                flipud=0.5, fliplr=0.5, patience=epochs, save_period=-1, val=True)
    best = out_dir / "yolo" / f"fold{fold}" / "weights" / "best.pt"
    if not best.exists():
        best = out_dir / "yolo" / f"fold{fold}" / "weights" / "last.pt"
    dst = out_dir / f"fold{fold}_yolo.pt"
    shutil.copy2(best, dst)
    # ultralytics may leave deterministic/cudnn flags behind; the refiner wants the defaults
    torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = False
    shutil.rmtree(out_dir / "yolo" / f"fold{fold}", ignore_errors=True)
    return dst


@torch.no_grad()
def predict_boxes(weights, files: list[str], size: int, split: str = "train",
                  conf: float = DET_CONF_MIN, augment: bool = False, nms_iou: float = DET_NMS_IOU,
                  model=None) -> dict[str, np.ndarray]:
    """{file: float32 (n, 5) [x0, y0, x1, y1, conf] in native px, conf-descending}.
    augment=True: ultralytics flip/scale TTA. nms_iou: NMS overlap threshold.
    RF-DETR weights (*_rfdetr.pth) are dispatched to the RF-DETR path (no NMS, no TTA)."""
    if str(weights).endswith("_rfdetr.pth"):
        return _predict_rfdetr(weights, files, split, conf, model)
    from ultralytics import YOLO
    model = model if model is not None else YOLO(str(weights))
    paths = [str(image_source(f, size, split)) for f in files]
    scale = 1.0 if DET_SOURCE == "native" else H / size
    out = {}
    for i in range(0, len(paths), 16):
        res = model.predict(source=paths[i:i + 16], imgsz=DET_IMGSZ, conf=conf, iou=nms_iou,
                            max_det=DET_MAX_DET, verbose=False, device=_device_arg(), augment=augment)
        for f, r in zip(files[i:i + 16], res):
            if r.boxes is None or len(r.boxes) == 0:
                out[f] = np.zeros((0, 5), np.float32)
                continue
            xyxy = r.boxes.xyxy.float().cpu().numpy() * scale
            c = r.boxes.conf.float().cpu().numpy()
            b = np.concatenate([xyxy, c[:, None]], 1).astype(np.float32)
            out[f] = b[np.argsort(-b[:, 4])]
    return out


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """a (n,4), b (m,4) xyxy -> (n,m)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    ix0 = np.maximum(a[:, None, 0], b[None, :, 0]); iy0 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix1 = np.minimum(a[:, None, 2], b[None, :, 2]); iy1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix1 - ix0, 0, None) * np.clip(iy1 - iy0, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + bb[None, :] - inter, 1e-6)


def fuse_boxes(box_sets: list[np.ndarray], iou_thr: float = DET_FUSE_IOU) -> np.ndarray:
    """Weighted boxes fusion over the fold detectors of one image. Boxes of all models are
    clustered greedily (conf-descending) at IoU > iou_thr against the running fused box;
    fused box = conf-weighted mean, fused conf = sum over models of their best conf / n_models,
    so a box only one of five detectors sees is down-weighted (fold agreement).
    Returns (n, 5) conf-descending; a single set is returned unchanged."""
    n_models = len(box_sets)
    if n_models == 1:
        return box_sets[0]
    rows = [np.concatenate([b, np.full((len(b), 1), m, np.float32)], 1) for m, b in enumerate(box_sets) if len(b)]
    if not rows:
        return np.zeros((0, 5), np.float32)
    allb = np.concatenate(rows)
    allb = allb[np.argsort(-allb[:, 4])]
    fused, members = [], []
    for b in allb:
        if fused:
            ious = box_iou(b[None, :4], np.array([f[:4] for f in fused]))[0]
            j = int(ious.argmax())
            if ious[j] > iou_thr:
                members[j].append(b)
                ms = np.array(members[j])
                wsum = ms[:, 4].sum()
                fused[j][:4] = (ms[:, :4] * ms[:, 4:5]).sum(0) / wsum
                continue
        fused.append(b[:5].copy())
        members.append([b])
    out = []
    for f, ms in zip(fused, members):
        ms = np.array(ms)
        best_per_model = {}
        for r in ms:
            best_per_model[int(r[5])] = max(best_per_model.get(int(r[5]), 0.0), float(r[4]))
        out.append([*f[:4], sum(best_per_model.values()) / n_models])
    out = np.array(out, np.float32)
    return out[np.argsort(-out[:, 4])]


def box_pr(pred: dict[str, np.ndarray], meta: dict, gt_rles: dict, conf: float,
           iou_thr: float = 0.5) -> dict:
    """Box-level precision/recall against every annotator record (greedy 1-1 at IoU > iou_thr).
    Diagnostic only: says whether the detector or the refiner is the bottleneck."""
    tp = fp = fn = 0
    for f, b in pred.items():
        b = b[b[:, 4] >= conf]
        for rid in meta["records_by_file"][f]:
            g = np.array([box_of_rle(r) for r in gt_rles[rid]], np.float32).reshape(-1, 4)
            g[:, 2:] += g[:, :2]
            m = box_iou(b[:, :4], g)
            used_g = set()
            hit = 0
            for i in np.argsort(-b[:, 4]) if len(b) else []:
                order = np.argsort(-m[i])
                for j in order:
                    if m[i, j] <= iou_thr:
                        break
                    if j not in used_g:
                        used_g.add(j); hit += 1
                        break
            tp += hit; fp += len(b) - hit; fn += len(g) - hit
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return {"conf": conf, "precision": p, "recall": r, "tp": tp, "fp": fp, "fn": fn}


In [ ]:
# === src/refiner.py ===
"""Rung 4, stage 2: per-box crop refiner.

Input  : 3 channels at native resolution, [image crop, CLAHE(image crop), seed], where the
         seed is the (padded) detector box filled with 1s - it tells the network WHICH
         filament to segment when several share the crop.
Target : the mask of that one annotation. Every (record, annotation) pair of the training
         files is a sample (all annotators: label augmentation, more data).
Crops  : side = max(REF_CROP, REF_CONTEXT x long side of the box) native px, centred on the
         box; only crops larger than REF_CROP are downscaled (INTER_AREA), and their
         probability map is upscaled back before thresholding - masks are never resampled.
Train-time box jitter (shift, scale, pad) mimics detector boxes; val uses clean GT boxes.
"""
from __future__ import annotations

import math
import random

import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch
from pycocotools import mask as mask_util
from torch.utils.data import Dataset



def crop_bounds(box_xywh, crop: int = REF_CROP, context: float = REF_CONTEXT) -> tuple[int, int, int]:
    """(x0, y0, side): square crop inside the frame, centred on the box."""
    x, y, w, h = box_xywh
    side = int(min(max(crop, math.ceil(context * max(w, h))), min(H, W)))
    x0 = int(round(x + w / 2 - side / 2)); y0 = int(round(y + h / 2 - side / 2))
    x0 = int(np.clip(x0, 0, W - side)); y0 = int(np.clip(y0, 0, H - side))
    return x0, y0, side


def seed_mask(box_xywh, x0: int, y0: int, side: int, pad: float = REF_PAD) -> np.ndarray:
    """uint8 (side, side): the box grown by `pad` x its size (at least 4 px) per side, filled."""
    x, y, w, h = box_xywh
    px, py = max(4.0, w * pad), max(4.0, h * pad)
    m = np.zeros((side, side), np.uint8)
    ax0, ay0 = int(np.clip(math.floor(x - px) - x0, 0, side)), int(np.clip(math.floor(y - py) - y0, 0, side))
    ax1, ay1 = int(np.clip(math.ceil(x + w + px) - x0, 0, side)), int(np.clip(math.ceil(y + h + py) - y0, 0, side))
    m[ay0:ay1, ax0:ax1] = 1
    return m


_CLAHE = None


def clahe(img: np.ndarray) -> np.ndarray:
    global _CLAHE
    if _CLAHE is None:
        _CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return _CLAHE.apply(img)


def make_input(img_crop: np.ndarray, seed: np.ndarray, crop: int = REF_CROP) -> np.ndarray:
    """uint8 image crop (side, side) + uint8 seed -> float32 (3, crop, crop)."""
    if img_crop.shape[0] != crop:
        img_crop = cv2.resize(img_crop, (crop, crop), interpolation=cv2.INTER_AREA)
        seed = cv2.resize(seed, (crop, crop), interpolation=cv2.INTER_NEAREST)
    x = np.stack([img_crop, clahe(img_crop), seed * 255]).astype(np.float32) / 255.0
    return (x - 0.5) / 0.25


def jitter_box(box_xywh, rng: random.Random) -> tuple[list[float], float]:
    """Perturb a GT box so training seeds look like detector boxes: shift up to 15 % of the
    size, scale 0.8-1.25 per axis, pad 0.5-1.5 x REF_PAD."""
    x, y, w, h = box_xywh
    x += rng.uniform(-0.15, 0.15) * w; y += rng.uniform(-0.15, 0.15) * h
    sx, sy = rng.uniform(0.8, 1.25), rng.uniform(0.8, 1.25)
    cx, cy = x + w / 2, y + h / 2
    w, h = max(6.0, w * sx), max(6.0, h * sy)
    return [cx - w / 2, cy - h / 2, w, h], rng.uniform(0.5, 1.5) * REF_PAD


def read_native(file: str, split: str = "train") -> np.ndarray:
    d = train_img_dir() if split == "train" else test_img_dir()
    im = cv2.imread(str(d / file), cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise FileNotFoundError(d / file)
    if im.shape != (H, W):
        im = cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)
    return im


class RefinerDataset(Dataset):
    """One sample per (annotator record, filament) of the given files, plus - when
    `negatives` is given ({file: (n, 4) xyxy boxes of detector false positives}) - hard
    negative samples: a detector FP box as the seed and an all-zero target, drawn so that
    negatives are `neg_frac` of the samples (sampled with replacement if there are fewer)."""

    def __init__(self, files: list[str], meta: dict, gt_rles: dict, augment: bool, crop: int = REF_CROP,
                 negatives: dict | None = None, neg_frac: float = REF_NEG_FRAC):
        self.items = []
        for f in files:
            for rid in meta["records_by_file"][f]:
                for rle in gt_rles[rid]:
                    b = box_of_rle(rle)
                    if b[2] >= 1 and b[3] >= 1:
                        self.items.append((f, rle, b))
        self.n_pos = len(self.items)
        if negatives:
            pool = [(f, None, (float(x0), float(y0), float(x1 - x0), float(y1 - y0)))
                    for f in files for x0, y0, x1, y1 in negatives.get(f, [])]
            if pool and neg_frac > 0:
                n_neg = int(round(self.n_pos * neg_frac / (1 - neg_frac)))
                rng = random.Random(0)
                self.items += [pool[i] for i in (rng.choices(range(len(pool)), k=n_neg) if n_neg > len(pool)
                                                 else rng.sample(range(len(pool)), n_neg))]
        self.n_neg = len(self.items) - self.n_pos
        self.augment, self.crop = augment, crop

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        f, rle, box = self.items[i]
        rng = random.Random(random.getrandbits(32)) if self.augment else random.Random(i)
        img = read_native(f)
        gt = mask_util.decode(rle) if rle is not None else np.zeros((H, W), np.uint8)
        seed_box, pad = (jitter_box(box, rng) if self.augment else (list(box), REF_PAD))
        x0, y0, side = crop_bounds(seed_box, self.crop)
        im = img[y0:y0 + side, x0:x0 + side]
        tgt = gt[y0:y0 + side, x0:x0 + side]
        seed = seed_mask(seed_box, x0, y0, side, pad)
        if side != self.crop:
            tgt = (cv2.resize(tgt.astype(np.float32), (self.crop, self.crop), interpolation=cv2.INTER_AREA) >= 0.5)
        tgt = tgt.astype(np.uint8)
        if self.augment:
            # photometric on the raw crop (before CLAHE, which is part of the input transform)
            x = im.astype(np.float32) / 255.0
            if rng.random() < 0.8:
                x = np.clip(x, 0, 1) ** rng.uniform(0.7, 1.4)
            if rng.random() < 0.8:
                x = x * rng.uniform(0.8, 1.2) + rng.uniform(-0.08, 0.08)
            if rng.random() < 0.3:
                x = x + np.random.normal(0, rng.uniform(0.005, 0.03), x.shape).astype(np.float32)
            im = (np.clip(x, 0, 1) * 255).astype(np.uint8)
        x = make_input(im, seed, self.crop)
        y = tgt[None].astype(np.float32)
        if self.augment:  # same geometric op on all channels and the target
            if rng.random() < 0.5:
                x, y = x[:, :, ::-1], y[:, :, ::-1]
            if rng.random() < 0.5:
                x, y = x[:, ::-1], y[:, ::-1]
            k = rng.randrange(4)
            x, y = np.rot90(x, k, axes=(1, 2)), np.rot90(y, k, axes=(1, 2))
        return (torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y)),
                torch.tensor(0.0 if rle is None else 1.0))


def build_refiner(encoder: str = REF_ENCODER, pretrained: bool = True, obj_head: bool = REF_OBJ_HEAD):
    """U-Net; with obj_head an auxiliary per-crop "objectness" head (global-pooled encoder
    features -> one logit, v8: WASH) and forward -> (mask, obj), else forward -> mask."""
    return smp.Unet(encoder_name=encoder, encoder_weights="imagenet" if pretrained else None,
                    in_channels=3, classes=1,
                    aux_params=dict(classes=1, pooling="avg", dropout=0.0) if obj_head else None)


def split_out(out):
    """(mask logits, obj logits or None) for either refiner variant."""
    return out if isinstance(out, (tuple, list)) else (out, None)


def save_refiner(path, model, **meta):
    torch.save({"state_dict": model.state_dict(), "kind": "refiner", **meta}, path)


def load_refiner(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    model = build_refiner(ck.get("encoder", REF_ENCODER), pretrained=False, obj_head=ck.get("obj_head", True)).to(device)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    return model, ck


@torch.no_grad()
def refine_boxes(models: list, img: np.ndarray, boxes: np.ndarray, device, tta: bool = True,
                 crop: int = REF_CROP, batch: int = 32) -> list[dict]:
    """boxes (n, 5) xyxy+conf native -> one crop record per box:
    {"prob": uint8 (crop, crop), "x0", "y0", "side", "conf", "box": xyxy, "obj": objectness in
    [0, 1]}. Probs and obj are the mean over models (and 4 flips if tta). Nothing is thresholded here: the crop records are what
    the OOF sweep and the offline sweeps consume."""
    recs, xs = [], []
    for b in boxes:
        x0b, y0b, x1b, y1b, conf = [float(v) for v in b[:5]]
        box = [x0b, y0b, max(1.0, x1b - x0b), max(1.0, y1b - y0b)]
        x0, y0, side = crop_bounds(box, crop)
        seed = seed_mask(box, x0, y0, side)
        xs.append(make_input(img[y0:y0 + side, x0:x0 + side], seed, crop))
        recs.append({"x0": x0, "y0": y0, "side": side, "conf": conf, "box": [x0b, y0b, x1b, y1b]})
    if not recs:
        return recs
    probs = np.zeros((len(recs), crop, crop), np.float32)
    objs = np.zeros(len(recs), np.float32)
    for i in range(0, len(xs), batch):
        x = torch.from_numpy(np.stack(xs[i:i + batch])).to(device)
        acc = torch.zeros((x.shape[0], 1, crop, crop), device=device)
        ob = torch.zeros(x.shape[0], device=device)
        views = [(x, None)]
        if tta:
            views += [(x.flip(-1), (-1,)), (x.flip(-2), (-2,)), (x.flip(-1, -2), (-1, -2))]
        for m in models:
            for v, dims in views:
                mk, o = split_out(m(v))
                acc += torch.sigmoid(mk).flip(dims) if dims else torch.sigmoid(mk)
                ob += torch.sigmoid(o[:, 0]) if o is not None else 1.0
        n = len(models) * len(views)
        probs[i:i + batch] = (acc / n)[:, 0].float().cpu().numpy()
        objs[i:i + batch] = (ob / n).float().cpu().numpy()
    for r, p, o in zip(recs, probs, objs):
        r["prob"] = np.round(p * 255).astype(np.uint8)
        r["obj"] = float(o)
    return recs


In [ ]:
# === src/cascade.py ===
"""Rung 4 orchestration: YOLO detector -> crop refiner, per grouped fold.

Per fold: train the detector on the fold's train files -> boxes for the val files (OOF) and
the test files; train the refiner on every (record, filament) of the train files -> refine
the OOF boxes into crop probability records. The OOF sweep then picks (detector conf cut,
refiner threshold, min_area) by pooled PQ over all annotator records. Test: boxes of the
fold detectors are fused (WBF), refined by the refiner ensemble, thresholded with the
selected knobs. Crop records (uint8 probs + bounds) are saved for OOF and test so every
post-processing knob can be re-swept offline (scripts/sweep_cascade.py).

Instances are assembled confidence-first at native resolution: a refined mask that mostly
lies on pixels already taken is a duplicate detection and is dropped; otherwise it is
clipped, so no two instances ever share a pixel.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader



# --- crop records <-> instances ---------------------------------------------------------------
def crop_instances(crops: list[dict], conf_thr: float, thr: float, min_area: int,
                   dup_fraction: float = DUP_FRACTION) -> list[np.ndarray]:
    """Crop records of one image -> non-overlapping uint8 (H, W) masks, confidence order."""
    taken = np.zeros((H, W), bool)
    out = []
    for c in sorted(crops, key=lambda c: -c["conf"]):
        if c["conf"] < conf_thr:
            continue
        x0, y0, side = int(c["x0"]), int(c["y0"]), int(c["side"])
        p = c["prob"].astype(np.float32) / 255.0
        if p.shape[0] != side:
            p = cv2.resize(p, (side, side), interpolation=cv2.INTER_LINEAR)
        b = (p >= thr).astype(np.uint8)
        if not b.any():
            continue
        # keep only the components that touch the seed box: the crop may contain neighbours
        n, labels = cv2.connectedComponents(b, connectivity=8)
        bx0, by0, bx1, by1 = c["box"]
        seed = seed_mask([bx0, by0, bx1 - bx0, by1 - by0], x0, y0, side).astype(bool)
        keep = np.unique(labels[seed & (labels > 0)])
        if len(keep) == 0:
            continue
        m = np.isin(labels, keep)
        area = int(m.sum())
        if area < min_area:
            continue
        t = taken[y0:y0 + side, x0:x0 + side]
        if (m & t).sum() > dup_fraction * area:
            continue
        m &= ~t
        if m.sum() < min_area:
            continue
        t |= m
        full = np.zeros((H, W), np.uint8)
        full[y0:y0 + side, x0:x0 + side] = m
        out.append(full)
    return out


def crop_instances_never_empty(crops: list[dict], conf_thr: float, thr: float, min_area: int) -> list[np.ndarray]:
    inst = crop_instances(crops, conf_thr, thr, min_area)
    if not inst and crops:
        inst = crop_instances(crops, 0.0, thr * 0.5, max(min_area // 3, 10))[:1]
    if not inst:
        m = np.zeros((H, W), np.uint8)
        m[1020:1024, 1020:1024] = 1
        inst = [m]
    return inst


def rescored(crops: list[dict], b: float) -> list[dict]:
    """Instance score = detector conf x objectness^b (b = 0: detector conf alone)."""
    if b == 0:
        return crops
    return [dict(c, conf=c["conf"] * max(c.get("obj", 1.0), 1e-3) ** b) for c in crops]


def score_crops(crops_by_file: dict, meta: dict, gt_rles: dict, conf_thr: float, thr: float,
                min_area: int) -> dict:
    acc = PQAccumulator()
    for f, crops in crops_by_file.items():
        preds = masks_to_rles(crop_instances(crops, conf_thr, thr, min_area))
        for rid in meta["records_by_file"][f]:
            acc.add(gt_rles[rid], preds)
    return acc.summary()


def sweep_cascade(oof: dict, meta: dict, gt_rles: dict) -> tuple[float, float, int, list]:
    table = []
    for conf in DET_CONFS:
        for t in REF_THRS:
            for a in REF_MIN_AREAS:
                s = score_crops(oof, meta, gt_rles, conf, t, a)
                table.append({"conf": conf, "thr": t, "min_area": a, **s})
                log(f"sweep conf={conf} thr={t} min_area={a}: PQ {s['pq']:.4f} SQ {s['sq']:.3f} "
                    f"RQ {s['rq']:.3f} tp {s['tp']} fp {s['fp']} fn {s['fn']}")
    best = max(table, key=lambda r: r["pq"])
    return best["conf"], best["thr"], best["min_area"], table


def save_crops(crops_by_file: dict, d: Path):
    """One compressed npz per file: probs (n, crop, crop) uint8 + meta (n, 9) float32
    [x0, y0, side, conf, bx0, by0, bx1, by1, obj]."""
    d.mkdir(exist_ok=True, parents=True)
    for f, crops in crops_by_file.items():
        stem = f.rsplit(".", 1)[0]
        probs = np.stack([c["prob"] for c in crops]) if crops else np.zeros((0, REF_CROP, REF_CROP), np.uint8)
        meta = np.array([[c["x0"], c["y0"], c["side"], c["conf"], *c["box"], c.get("obj", 1.0)] for c in crops],
                        np.float32).reshape(-1, 9)
        np.savez_compressed(d / f"{stem}.npz", probs=probs, meta=meta)


def load_crops(d: Path) -> dict:
    out = {}
    for p in sorted(Path(d).glob("*.npz")):
        z = np.load(p)
        out[p.stem + ".jpeg"] = [{"prob": pr, "x0": m[0], "y0": m[1], "side": m[2], "conf": float(m[3]),
                                  "box": list(m[4:8]), "obj": float(m[8]) if len(m) > 8 else 1.0}
                                 for pr, m in zip(z["probs"], z["meta"])]
    return out


def write_cascade_submission(test_crops: dict, conf: float, thr: float, min_area: int, out_dir: Path):
    rows = []
    for f, crops in test_crops.items():
        stem = f.rsplit(".", 1)[0]
        for k, m in enumerate(crop_instances_never_empty(crops, conf, thr, min_area), 1):
            rows.append({"filament_id": f"{stem}_{k}", "segmentation_rle": mask_to_rle(m)})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    assert df.filament_id.is_unique and len(df) >= len(test_crops)
    df.to_csv(out_dir / "submission.csv", index=False)
    log(f"submission.csv: {len(df)} rows over {len(test_crops)} images (conf={conf}, thr={thr}, min_area={min_area})")
    return df


# --- hard negatives ---------------------------------------------------------------------------
def hard_negatives(oof_boxes: dict, meta: dict, gt_rles: dict, conf: float = REF_NEG_CONF,
                   iou: float = REF_NEG_IOU) -> dict:
    """{file: (n, 4) xyxy} detector OOF boxes above `conf` that overlap no GT box of any
    annotator record of the file by more than `iou`: false positives by every annotator."""
    out = {}
    for f, b in oof_boxes.items():
        b = b[b[:, 4] >= conf]
        if not len(b):
            continue
        g = np.array([box_of_rle(r) for rid in meta["records_by_file"][f] for r in gt_rles[rid]],
                     np.float32).reshape(-1, 4)
        g[:, 2:] += g[:, :2]
        m = box_iou(b[:, :4], g)
        keep = (m.max(1) < iou) if len(g) else np.ones(len(b), bool)
        if keep.any():
            out[f] = b[keep, :4]
    return out


@torch.no_grad()
def obj_separation(model, ds, device, n: int = 400) -> dict:
    """Objectness head on val crops: mean sigmoid on positives / negatives, fraction of
    negatives rejected (obj < 0.5) and of negatives with a < 100 px mask at thr 0.5."""
    model.eval()
    out = {}
    for name, idx in (("pos", list(range(0, min(ds.n_pos, n)))),
                      ("neg", list(range(ds.n_pos, min(len(ds.items), ds.n_pos + n))))):
        if not idx:
            continue
        objs, small = [], 0
        for i in range(0, len(idx), 32):
            x = torch.stack([ds[j][0] for j in idx[i:i + 32]]).to(device)
            mk, o = split_out(model(x))
            objs.append(torch.sigmoid(o[:, 0]).cpu() if o is not None else torch.ones(x.shape[0]))
            small += int(((torch.sigmoid(mk) >= 0.5).sum(dim=(1, 2, 3)) < 100).sum())
        o = torch.cat(objs)
        out[f"{name}_obj"] = float(o.mean())
        out[f"{name}_reject"] = float((o < 0.5).float().mean())
        out[f"{name}_small"] = small / len(idx)
    return out


# --- refiner training -------------------------------------------------------------------------
def refiner_loss(logits, obj, y, is_pos):
    """Positives: pixel BCE + soft Dice; negatives (all-zero target): pixel BCE only — Dice on
    an empty target has a ~1/area^2 gradient and taught nothing in v7. Plus BCE on the
    objectness logit vs the positive/negative label."""
    bce = F.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    inter = (p * y).sum(dim=(1, 2, 3))
    den = p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + 1) / (den + 1)
    dice = (dice * is_pos).sum() / is_pos.sum().clamp(min=1)
    if obj is None:
        return bce + dice
    return bce + dice + F.binary_cross_entropy_with_logits(obj[:, 0], is_pos)


@torch.no_grad()
def val_dice(model, dl, device) -> float:
    model.eval()
    num = den = 0.0
    for x, y, _ in dl:
        x, y = x.to(device), y.to(device)
        p = (torch.sigmoid(split_out(model(x))[0]) >= 0.5).float()
        num += float((2 * (p * y).sum(dim=(1, 2, 3))).sum())
        den += float((p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3)) + 1e-6).sum())
    return num / max(den, 1e-6)


def train_refiner(fold: int, meta: dict, gt_rles: dict, device, out_dir: Path, epochs: int = REF_EPOCHS,
                  max_steps: int = MAX_STEPS, eval_files: int = EVAL_FILES, negatives: dict | None = None) -> Path:
    folds = meta["folds"]
    tr = sorted(f for f, k in folds.items() if k != fold)
    va = sorted(f for f, k in folds.items() if k == fold)
    if eval_files:
        va = va[:eval_files]
        if DEBUG:
            tr = tr[:max(eval_files * 2, 4)]
    ds_tr = RefinerDataset(tr, meta, gt_rles, augment=True, negatives=negatives)
    ds_va = RefinerDataset(va, meta, gt_rles, augment=False)
    ds_va_neg = RefinerDataset(va, meta, gt_rles, augment=False, negatives=negatives, neg_frac=0.5) if negatives else None
    log(f"fold {fold}: refiner {REF_ENCODER} crops {ds_tr.n_pos} pos + {ds_tr.n_neg} neg train / "
        f"{len(ds_va)} val (+{ds_va_neg.n_neg if ds_va_neg else 0} val neg), {epochs} epochs")
    dl = DataLoader(ds_tr, batch_size=REF_BATCH, shuffle=True, num_workers=WORKERS, drop_last=True,
                    persistent_workers=WORKERS > 0)
    dl_va = DataLoader(ds_va, batch_size=REF_BATCH, shuffle=False, num_workers=min(WORKERS, 2))
    model = build_refiner(REF_ENCODER, pretrained=not DEBUG).to(device)
    steps = min(len(dl), max_steps) if max_steps else len(dl)
    opt = torch.optim.AdamW(model.parameters(), lr=REF_LR, weight_decay=WEIGHT_DECAY)
    # OneCycleLR divides by (warm-up steps - 1): with a 10-step debug epoch pct_start 0.1 is 0/0
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=REF_LR, epochs=epochs, steps_per_epoch=steps,
                                                pct_start=0.1 if epochs * steps >= 30 else 0.3)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ckpt, best, history = out_dir / f"fold{fold}_refiner.pt", -1.0, []
    for epoch in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for step, (x, y, is_pos) in enumerate(dl):
            if step >= steps:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            is_pos = is_pos.to(device, non_blocking=True)
            with torch.autocast("cuda", enabled=use_amp):
                logits, obj = split_out(model(x))
                loss = refiner_loss(logits.float(), obj.float() if obj is not None else None, y, is_pos)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item(); n += 1
        d = val_dice(model, dl_va, device)
        sep = obj_separation(model, ds_va_neg, device) if ds_va_neg else {}
        history.append({"epoch": epoch, "loss": tot / max(n, 1), "val_dice": d, **sep,
                        "time": time.time() - t0})
        log(f"fold {fold} refiner epoch {epoch:2d} loss {tot / max(n, 1):.4f} | val dice {d:.4f} | "
            f"obj pos {sep.get('pos_obj', float('nan')):.3f} neg {sep.get('neg_obj', float('nan')):.3f} | "
            f"neg reject {sep.get('neg_reject', float('nan')):.3f} small {sep.get('neg_small', float('nan')):.3f} | "
            f"{time.time() - t0:.0f}s")
        if d > best:
            best = d
            save_refiner(ckpt, model, encoder=REF_ENCODER, crop=REF_CROP, fold=fold, epoch=epoch, dice=best,
                         obj_head=REF_OBJ_HEAD)
        json.dump(history, open(out_dir / f"fold{fold}_refiner_history.json", "w"), indent=1)
    return ckpt


# --- inference helpers ------------------------------------------------------------------------
def refine_files(refiners: list, boxes_by_file: dict, files: list[str], split: str, device,
                 tta: bool) -> dict:
    out = {}
    for f in files:
        b = boxes_by_file.get(f, np.zeros((0, 5), np.float32))
        b = b[b[:, 4] >= DET_CONF_MIN]
        out[f] = refine_boxes(refiners, read_native(f, split), b, device, tta=tta)
    return out


def fused_test_boxes(test_boxes: list[dict]) -> dict:
    files = test_files()
    return {f: fuse_boxes([tb.get(f, np.zeros((0, 5), np.float32)) for tb in test_boxes]) for f in files}


def find_attached_cascade() -> tuple[list[Path], list[Path]]:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return [], []
    y = sorted(set(base.glob("*/fold*_yolo.pt")) | set(base.glob("*/*/fold*_yolo.pt")) |
               set(base.glob("*/fold*_rfdetr.pth")) | set(base.glob("*/*/fold*_rfdetr.pth")))
    r = sorted(set(base.glob("*/fold*_refiner.pt")) | set(base.glob("*/*/fold*_refiner.pt")))
    return y, r


# --- entry ------------------------------------------------------------------------------------
def main_cascade(meta: dict, gt_rles: dict, out_dir: Path):
    device = get_device()
    build_yolo_labels(meta, gt_rles)
    tfiles = test_files()
    if EVAL_FILES and DEBUG:
        tfiles = tfiles[:EVAL_FILES]

    yolos, refs = find_attached_cascade()
    if yolos and refs:
        log(f"inference mode: {len(yolos)} detectors + {len(refs)} refiners attached")
        refiners, cks = [], []
        for p in refs:
            m, ck = load_refiner(p, device)
            refiners.append(m); cks.append(ck)
        conf = float(np.median([ck.get("conf", 0.3) for ck in cks]))
        thr = float(np.median([ck.get("thr", 0.5) for ck in cks]))
        area = int(np.median([ck.get("min_area", 100) for ck in cks]))
        tb = [predict_boxes(p, tfiles, meta["size"], split="test") for p in yolos]
        crops = refine_files(refiners, fused_test_boxes(tb), tfiles, "test", device, TTA)
        save_crops(crops, out_dir / "test_crops")
        write_cascade_submission(crops, conf, thr, area, out_dir)
        return

    max_folds = N_FOLDS if MAX_FOLDS == "auto" else int(MAX_FOLDS)
    train_limit = max(EVAL_FILES * 2, 4) if (DEBUG and EVAL_FILES) else 0
    val_files = {}
    for fold in range(N_FOLDS):
        va = sorted(f for f, k in meta["folds"].items() if k == fold)
        val_files[fold] = va[:EVAL_FILES] if EVAL_FILES else va

    # phase 1: detectors. Every fold's detector first, because the refiners (phase 2) train on
    # the detectors' OOF false positives as hard negatives (REF_NEG_FRAC > 0).
    det_folds, test_boxes, oof_boxes_all, box_prs, det_time = [], [], {}, {}, 0.0
    for fold in range(START_FOLD, N_FOLDS):
        if len(det_folds) >= max_folds:
            break
        elapsed = time.time() - T0
        if det_folds and elapsed + 1.15 * det_time > TIME_BUDGET / 2:
            log(f"time budget (detectors): {elapsed / 60:.0f} min used -> stop after {len(det_folds)} folds")
            break
        t0 = time.time()
        det = train_detector(fold, meta, out_dir, train_limit=train_limit, val_limit=EVAL_FILES)
        ob = predict_boxes(det, val_files[fold], meta["size"])
        pr = {c: box_pr(ob, meta, gt_rles, c) for c in (0.2, 0.3, 0.5)}
        for c, r in pr.items():
            log(f"fold {fold} boxes conf>={c}: P {r['precision']:.3f} R {r['recall']:.3f} "
                f"(tp {r['tp']} fp {r['fp']} fn {r['fn']})")
        test_boxes.append(predict_boxes(det, tfiles, meta["size"], split="test"))
        json.dump({f: b.tolist() for f, b in ob.items()}, open(out_dir / f"fold{fold}_oof_boxes.json", "w"))
        oof_boxes_all.update(ob)
        box_prs[fold] = pr
        det_folds.append(fold)
        det_time = time.time() - t0
    negatives = hard_negatives(oof_boxes_all, meta, gt_rles) if REF_NEG_FRAC > 0 else None
    if negatives:
        log(f"hard negatives: {sum(len(v) for v in negatives.values())} detector FP boxes over "
            f"{len(negatives)} files (conf >= {REF_NEG_CONF}, IoU < {REF_NEG_IOU} to every annotator)")

    # phase 2: refiners, OOF crops, scoreable submission after every fold
    results, ref_ckpts, fold_time = [], [], 0.0
    oof = {}
    for fold in det_folds:
        elapsed = time.time() - T0
        if results and elapsed + 1.15 * fold_time > TIME_BUDGET:
            log(f"time budget: {elapsed / 60:.0f} min used, fold takes {fold_time / 60:.0f} min -> stop")
            break
        t0 = time.time()
        va = val_files[fold]
        oof_boxes = {f: oof_boxes_all[f] for f in va}
        ck = train_refiner(fold, meta, gt_rles, device, out_dir, negatives=negatives)
        ref_ckpts.append(ck)
        refiner, _ = load_refiner(ck, device)
        crops = refine_files([refiner], oof_boxes, va, "train", device, TTA)
        save_crops(crops, out_dir / "oof_crops")
        oof.update(crops)
        s = score_crops(crops, meta, gt_rles, 0.3, 0.5, 100)
        log(f"fold {fold} OOF cascade PQ {s['pq']:.4f} SQ {s['sq']:.3f} RQ {s['rq']:.3f} "
            f"tp {s['tp']} fp {s['fp']} fn {s['fn']} (conf 0.3 / thr 0.5 / min_area 100)")
        results.append({"fold": fold, "box_pr": box_prs[fold], "oof_pq": s["pq"], "refiner": str(ck)})
        del refiner
        torch.cuda.empty_cache() if device.type == "cuda" else None
        fold_time = time.time() - t0
        # keep the run scoreable: fused boxes of ALL trained detectors + refiners so far, default knobs
        refiners = [load_refiner(c, device)[0] for c in ref_ckpts]
        tcrops = refine_files(refiners, fused_test_boxes(test_boxes), tfiles, "test", device, tta=False)
        write_cascade_submission(tcrops, 0.3, 0.5, 100, out_dir)
        del refiners, tcrops
        torch.cuda.empty_cache() if device.type == "cuda" else None

    log(f"OOF sweep over {len(oof)} files from {len(results)} folds")
    conf, thr, area, table = sweep_cascade(oof, meta, gt_rles)
    best = max(table, key=lambda r: r["pq"])
    summary = {"pipeline": "cascade", "folds": results, "sweep": table, "best_conf": conf, "best_thr": thr,
               "best_min_area": area, "best_pq": best["pq"]}
    json.dump(summary, open(out_dir / "summary.json", "w"), indent=1)
    for c in ref_ckpts:  # stamp the selected knobs into the refiner checkpoints (inference mode reads them)
        ck = torch.load(c, map_location="cpu", weights_only=False)
        ck.update(conf=conf, thr=thr, min_area=area)
        torch.save(ck, c)
    refiners = [load_refiner(c, device)[0] for c in ref_ckpts]
    tcrops = refine_files(refiners, fused_test_boxes(test_boxes), tfiles, "test", device, TTA)
    save_crops(tcrops, out_dir / "test_crops")
    write_cascade_submission(tcrops, conf, thr, area, out_dir)
    log(f"OOF PQ at selected (conf={conf}, thr={thr}, min_area={area}): {best['pq']:.4f}")


In [ ]:
# === src/pipeline.py ===
"""Orchestrator. Same entry point locally and in the Kaggle kernel.

main():  placeholder submission -> cache -> grouped folds
         -> [PIPELINE == "cascade": hand over to cascade.main_cascade (rung 4) and return]
         -> [inference mode if fold*_best.pt checkpoints are attached under /kaggle/input]
         -> else train folds until TIME_BUDGET, saving fold{k}_best.pt + OOF prob maps,
            rewriting submission.csv after every fold so the run is always scoreable
         -> OOF sweep of (threshold, min_area) on all trained folds
         -> fold-ensemble test prediction -> submission.csv
run():   main() inside a crash guard.
"""
from __future__ import annotations

import json
import time
import traceback
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader



# --- always-scoreable ---------------------------------------------------------------------
def write_placeholder(out_dir: Path):
    rows = []
    for f in test_files():
        m = np.zeros((H, W), np.uint8)
        m[1020:1028, 1020:1028] = 1
        rows.append({"filament_id": f"{f.rsplit('.', 1)[0]}_1", "segmentation_rle": mask_to_rle(m)})
    pd.DataFrame(rows).to_csv(out_dir / "submission.csv", index=False)
    log(f"placeholder submission written: {len(rows)} rows")


def write_submission(test_probs: dict, threshold: float, min_area: int, out_dir: Path,
                     close_kernel: int = 0, min_score: float = 0.0) -> pd.DataFrame:
    rows = []
    for name, prob in test_probs.items():
        stem = name.rsplit(".", 1)[0]
        inst = instances_never_empty(prob, threshold, min_area, close_kernel, min_score)
        for k, m in enumerate(inst, 1):
            rows.append({"filament_id": f"{stem}_{k}", "segmentation_rle": mask_to_rle(m)})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    assert df.filament_id.is_unique and len(df) >= len(test_probs)
    df.to_csv(out_dir / "submission.csv", index=False)
    log(f"submission.csv: {len(df)} rows over {len(test_probs)} images "
        f"(thr={threshold}, min_area={min_area}, min_score={min_score})")
    return df


def save_test_probs(test_probs: dict, out_dir: Path):
    """Ensemble-mean test prob maps as uint8 PNGs: lets post-processing be re-done
    locally without another kernel run."""
    d = out_dir / "test"
    d.mkdir(exist_ok=True)
    for name, p in test_probs.items():
        cv2.imwrite(str(d / (name.rsplit(".", 1)[0] + ".png")), np.round(p * 255).astype(np.uint8))


# --- training -----------------------------------------------------------------------------
def train_fold(fold: int, meta: dict, device, out_dir: Path, gt_rles: dict, epochs: int = EPOCHS,
               max_steps: int = MAX_STEPS, eval_files: int = EVAL_FILES, arch: str = ARCH,
               encoder: str = ENCODER) -> dict:
    folds = meta["folds"]
    tr = sorted(f for f, k in folds.items() if k != fold)
    va = sorted(f for f, k in folds.items() if k == fold)
    if eval_files:
        va = va[:eval_files]
    log(f"fold {fold}: train {len(tr)} files, val {len(va)} files, {arch}/{encoder}")

    model = build_model(arch, encoder).to(device)
    ds = CropDataset(tr, meta, crop=CROP, samples_per_file=SAMPLES_PER_FILE)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=WORKERS, drop_last=True,
                    persistent_workers=WORKERS > 0)
    steps = min(len(dl), max_steps) if max_steps else len(dl)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=steps,
                                                pct_start=0.1)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ckpt = out_dir / f"fold{fold}_best.pt"
    best, history = -1.0, []
    for epoch in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for step, (x, y) in enumerate(dl):
            if step >= steps:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast("cuda", enabled=use_amp):
                loss = bce_dice_loss(model(x), y, pos_weight=POS_WEIGHT)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item(); n += 1
        res, _ = evaluate(model, va, meta, device, gt_rles, (EPOCH_THR,), MIN_AREA, tta=False)
        r = res[EPOCH_THR]
        rec = {"epoch": epoch, "loss": tot / max(n, 1), **r, "time": time.time() - t0}
        history.append(rec)
        log(f"fold {fold} epoch {epoch:2d} loss {rec['loss']:.4f} | PQ {r['pq']:.4f} SQ {r['sq']:.3f} "
            f"RQ {r['rq']:.3f} tp {r['tp']} fp {r['fp']} fn {r['fn']} | {rec['time']:.0f}s")
        if r["pq"] > best:
            best = r["pq"]
            save_checkpoint(ckpt, model, arch=arch, encoder=encoder, img_size=meta["size"], fold=fold,
                            epoch=epoch, pq=best)
        json.dump(history, open(out_dir / f"fold{fold}_history.json", "w"), indent=1)

    # OOF prob maps with the best epoch + TTA, saved as uint8 PNGs for offline sweeps
    model, _ = load_checkpoint(ckpt, device)
    res, probs = evaluate(model, va, meta, device, gt_rles, (EPOCH_THR,), MIN_AREA, tta=TTA,
                          keep_probs=True)
    oof_dir = out_dir / "oof"
    oof_dir.mkdir(exist_ok=True)
    for name, p in probs.items():
        cv2.imwrite(str(oof_dir / (name.rsplit(".", 1)[0] + ".png")), p)
    log(f"fold {fold} done: best epoch PQ {best:.4f}, TTA PQ {res[EPOCH_THR]['pq']:.4f}")
    return {"fold": fold, "ckpt": ckpt, "best_pq": best, "tta_pq": res[EPOCH_THR]["pq"],
            "oof": probs, "val_files": va, "model": model}


def sweep_postprocess(oof: dict, meta: dict, gt_rles: dict) -> tuple[float, int, float, list]:
    table = []
    for t in THRESHOLDS:
        for a in MIN_AREAS:
            for ms in MIN_SCORES:
                s = score_probs(oof, meta, gt_rles, t, a, CLOSE_KERNEL, ms)
                table.append({"thr": t, "min_area": a, "min_score": ms, "close": CLOSE_KERNEL, **s})
                log(f"sweep thr={t} min_area={a} min_score={ms} close={CLOSE_KERNEL}: PQ {s['pq']:.4f} "
                    f"SQ {s['sq']:.3f} RQ {s['rq']:.3f} tp {s['tp']} fp {s['fp']} fn {s['fn']}")
    best = max(table, key=lambda r: r["pq"])
    return best["thr"], best["min_area"], best["min_score"], table


def predict_test(models: list, size: int, device) -> dict:
    """Average prob maps of several models over the test set. {file: float32 (size,size)}."""
    files = test_files()
    acc = {}
    # float16: 180 maps at 2048^2 are 3 GB in float32; several such dicts coexist
    for m in models:
        for name, p in predict_files(m, files, size, device, split="test", tta=TTA):
            acc[name] = (acc[name] + p.astype(np.float16)) if name in acc else p.astype(np.float16)
    return {k: (v / len(models)).astype(np.float16) for k, v in acc.items()}


def find_attached_checkpoints() -> list[Path]:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return []
    return sorted(set(base.glob("*/fold*_best.pt")) | set(base.glob("*/*/fold*_best.pt")))


# --- entry ---------------------------------------------------------------------------------
# local runs get a "local-" prefix so they can never be confused with pulled kernel runs
OUT_DIR = RUNS if ON_KAGGLE else RUNS / time.strftime("local-%Y%m%d-%H%M%S")


def main():
    seed_everything(SEED)
    device = get_device()
    out_dir = OUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    log(f"device={device} on_kaggle={ON_KAGGLE} out={out_dir} cache={CACHE}")
    write_placeholder(out_dir)

    meta = build_cache(IMG_SIZE, N_FOLDS)
    gt_rles = load_gt_rles()

    if PIPELINE == "cascade":          # rung 4: detector -> crop refiner (cascade.py)
        main_cascade(meta, gt_rles, out_dir)
        return

    attached = find_attached_checkpoints()
    if attached:
        log(f"inference mode: {len(attached)} checkpoints attached")
        models, cks = [], []
        for p in attached:
            m, ck = load_checkpoint(p, device)
            models.append(m); cks.append(ck)
            log(f"  {p.name}: fold {ck.get('fold')} epoch {ck.get('epoch')} PQ {ck.get('pq', 0):.4f}")
        thr = float(np.median([ck.get("thr", EPOCH_THR) for ck in cks]))
        area = int(np.median([ck.get("min_area", MIN_AREA) for ck in cks]))
        ms = float(np.median([ck.get("min_score", 0.0) for ck in cks]))
        probs = predict_test(models, meta["size"], device)
        save_test_probs(probs, out_dir)
        write_submission(probs, thr, area, out_dir, CLOSE_KERNEL, ms)
        return

    # training mode
    max_folds = N_FOLDS if MAX_FOLDS == "auto" else int(MAX_FOLDS)
    results, test_sum, n_models, fold_time = [], None, 0, 0.0
    for fold in range(START_FOLD, N_FOLDS):
        if len(results) >= max_folds:
            break
        elapsed = time.time() - T0
        if results and elapsed + 1.15 * fold_time > TIME_BUDGET:
            log(f"time budget: {elapsed / 60:.0f} min used, fold takes {fold_time / 60:.0f} min -> stop")
            break
        t0 = time.time()
        r = train_fold(fold, meta, device, out_dir, gt_rles)
        fold_time = time.time() - t0
        results.append(r)
        # keep the run scoreable: ensemble of folds so far
        p = predict_test([r["model"]], meta["size"], device)
        test_sum = p if test_sum is None else {k: (test_sum[k] + p[k]).astype(np.float16) for k in p}
        del p
        n_models += 1
        write_submission({k: v / n_models for k, v in test_sum.items()}, EPOCH_THR, MIN_AREA, out_dir,
                         CLOSE_KERNEL)
        del r["model"]
        torch.cuda.empty_cache() if device.type == "cuda" else None

    oof = {}
    for r in results:
        oof.update(r["oof"])
    log(f"OOF sweep over {len(oof)} files from {len(results)} folds")
    thr, area, ms, table = sweep_postprocess(oof, meta, gt_rles)
    summary = {"folds": [{k: v for k, v in r.items() if k in ("fold", "best_pq", "tta_pq")} for r in results],
               "sweep": table, "best_thr": thr, "best_min_area": area, "best_min_score": ms}
    json.dump(summary, open(out_dir / "summary.json", "w"), indent=1)
    for r in results:  # stamp the selected post-processing into the checkpoints
        ck = torch.load(r["ckpt"], map_location="cpu", weights_only=False)
        ck.update(thr=thr, min_area=area, min_score=ms)
        torch.save(ck, r["ckpt"])
    test_mean = {k: v / n_models for k, v in test_sum.items()}
    save_test_probs(test_mean, out_dir)
    write_submission(test_mean, thr, area, out_dir, CLOSE_KERNEL, ms)
    log(f"OOF PQ at selected (thr={thr}, min_area={area}, min_score={ms}): "
        f"{max(table, key=lambda x: x['pq'])['pq']:.4f}")


def run():
    """Entry point with the crash guard: a placeholder submission exists from the first
    seconds of main(), so a crash still leaves a scoreable file."""
    try:
        main()
    except Exception:
        traceback.print_exc()
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        if not (OUT_DIR / "submission.csv").exists():
            write_placeholder(OUT_DIR)
        log("run FAILED; see traceback above")
    log("done")


In [ ]:
# --- run ------------------------------------------------------------------------------
run()
